In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/alexcristea72/logits/mrbert_biomed_seed42_epoch2_blind_logits.npy
/kaggle/input/datasets/alexcristea72/logits/mrbert_biomed_seed42_epoch3_blind_logits.npy
/kaggle/input/datasets/alexcristea72/logits/mrbert_biomed_seed42_epoch5_blind_logits.npy
/kaggle/input/datasets/alexcristea72/logits/mrbert_biomed_seed42_epoch4_blind_logits.npy
/kaggle/input/datasets/alexcristea72/logits/mrbert_biomed_seed42_epoch1_blind_logits.npy
/kaggle/input/datasets/alexcristea72/iberlef-grace/merged_train_dev_full_fit.json
/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json
/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json
/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_2_train.json
/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_2_dev.json
/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/track2_scoring_program/READM

In [4]:
!pip install sentencepiece

In [ ]:
# =========================================================================================
# 🩺 GRACE Track 1 — THE MEDICAL EXPERT (Real-Time Scoring + Seed Locked)
# =========================================================================================

import subprocess, sys
# 1. Download Spanish grammar engine silently
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import os
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score

# =========================================================================================
# 0. LOCK THE UNIVERSE (Seed Everything)
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Force deterministic operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# =========================================================================================
# PART 1: THE DATA PARSER (Prev-Sentence Context Included)
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    nlp = spacy.load("es_core_news_sm")
    records = []

    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]

        spacy_doc = nlp(raw_text)
        all_sents = list(spacy_doc.sents)
        sent_records = []  

        for ann in annotations:
            sent_records.append({"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]})

        for sent in all_sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": s})

        sent_records.sort(key=lambda x: x["start"])

        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": sr["label"]})

    return pd.DataFrame(records)


# =========================================================================================
# PART 2: LOAD & ENCODE
# =========================================================================================
train_df = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json")
dev_df   = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json")

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

train_df["label"] = train_df["label"].map(label_map)
dev_df["label"]   = dev_df["label"].map(label_map)

MODEL_NAME = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nDownloading Clinical RoBERTa weights...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True 
)

def tokenize_fn(examples):
    return tokenizer(examples["prev_text"], examples["text"], padding="max_length", truncation=True, max_length=192)

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)


# =========================================================================================
# PART 3: IN-MEMORY SCORING CACHE (Optimization)
# =========================================================================================
print("Pre-building Dev set evaluation template...")
nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json", "r", encoding="utf-8"))

dev_eval_template = []
for doc in dev_raw_data:
    raw_text = doc["raw_text"]
    annotations = doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    spacy_doc = nlp_cache(raw_text)

    sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
    
    for sent in spacy_doc.sents:
        s, e = sent.start_char, sent.end_char
        if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
            if len(sent.text.strip()) > 10:
                sent_records.append({"text": sent.text.strip(), "start": s, "end": e})

    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({
        "id": doc["id"],
        "raw_text": raw_text,
        "annotations": {"entities": annotations},
        "sent_records": sent_records 
    })


# =========================================================================================
# PART 4: THE OFFICIAL SCORING ENGINE (Stripped down to Strict rules)
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def _tokenize(text: str):
    return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]

def _span_token_set(token_positions, start, end):
    return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)

def _enrich(entities, token_positions):
    return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]

class CompTuple(NamedTuple):
    start: int; end: int; type: str; tokens: frozenset

def _exact_span(a, b):
    return a.type == b.type and a.start == b.start and a.end == b.end

def _greedy_match(items_a, items_b, match_fn):
    used = set()
    matched, unmatched_a = [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None:
            matched.append((a, items_b[j]))
            used.add(j)
        else:
            unmatched_a.append(a)
    return matched, unmatched_a, [items_b[j] for j in range(len(items_b)) if j not in used]

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return f

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    
    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))
        gold_ents = _enrich(case["annotations"]["entities"], tp)
        pred_ents = _enrich(case["predictions"]["entities"], tp)
        
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]

        m, ug, up = _greedy_match(gold_comps, pred_comps, _exact_span)
        for g, _ in m: tp_s[g.type] += 1
        for r in up:   fp_s[r.type] += 1
        for r in ug:   fn_s[r.type] += 1

    f1s = []
    metrics = {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        f = _prf(tp_s[t], fp_s[t], fn_s[t])
        metrics[t] = f
        f1s.append(f)

    metrics["Macro_F1"] = sum(f1s) / len(f1s) if f1s else 0.0
    return metrics


# =========================================================================================
# PART 5: THE LIVE EPOCH CALLBACK
# =========================================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    
    # 0.50 MajorClaim override applied in memory
    preds[probs[:, 3] > 0.50] = 3  

    text_to_pred = {text: pred for text, pred in zip(dev_df["text"], preds)}

    cases = []
    for template in dev_eval_template:
        predicted_entities = []
        e_id = 1
        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({
                    "id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label
                })
                e_id += 1

        cases.append({
            "raw_text": template["raw_text"],
            "annotations": template["annotations"],
            "predictions": {"entities": predicted_entities}
        })

    metrics = evaluate_strict_f1(cases)

    print(f"\n--- EPOCH EVALUATION ---")
    print(f"🥇 Macro F1   : {metrics['Macro_F1']:.4f}")
    print(f"📊 Premise    : {metrics['Premise']:.4f}")
    print(f"📈 Claim      : {metrics['Claim']:.4f}")
    print(f"👑 MajorClaim : {metrics['MajorClaim']:.4f}")
    print(f"------------------------\n")

    return {"strict_macro_f1": metrics["Macro_F1"]}


# =========================================================================================
# PART 6: TRAINING SETUP & DIFFERENTIAL OPTIMIZER
# =========================================================================================
counts  = np.bincount(train_df["label"].values, minlength=4).astype(float)
raw_w   = counts.sum() / (len(counts) * counts)
weights = (raw_w / raw_w.mean()).tolist()
weights[3] = 10.0  

# Split weights: slow learning for the pretrained body, fast learning for the new head
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
base_params = [p for n, p in model.named_parameters() if "classifier" not in n]

custom_optimizer = torch.optim.AdamW([
    {"params": base_params, "lr": 1e-5},
    {"params": head_params, "lr": 1e-4}
], weight_decay=0.05)

training_args = TrainingArguments(
    output_dir="./roberta_grace", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine",
    per_device_train_batch_size=16, per_device_eval_batch_size=32, num_train_epochs=15,           
    fp16=True, 
    load_best_model_at_end=True,                
    metric_for_best_model="strict_macro_f1",    
    greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=42,          # <--- Seed locked for shuffling
    data_seed=42      # <--- Seed locked for data loading
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        w       = torch.tensor(weights, dtype=logits.dtype, device=logits.device)
        loss    = F.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedCETrainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=dev_dataset,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    optimizers=(custom_optimizer, None)
)

print("\nStarting Real-Time Evaluation Training Loop...")
trainer.train()

# =========================================================================================
# PART 7: EXTRACTION OF THE GOLDEN LOGITS
# =========================================================================================

print("\nExtracting Golden Predictions for Ensemble...")
raw_preds = trainer.predict(dev_dataset)
logits = raw_preds.predictions

np.save("roberta_dev_logits.npy", logits)
print("✅ Saved roberta_dev_logits.npy (Best Epoch Captured)!")

In [ ]:
# =========================================================================================
# 🩺 GRACE Track 1 — THE MEDICAL EXPERT (Future-Sight + Seed Locked + Differential LR)
# 
# THEORETICAL FRAMEWORK:
# 1. BI-DIRECTIONAL CONTEXT: A sentence's role is often determined by its neighbors. 
#    A "Premise" provides evidence for a "Claim" that follows it. By feeding the AI 
#    both the PREVIOUS and NEXT sentences, we allow the Self-Attention mechanism 
#    to see the logical flow of the entire argument.
# 2. OPTIMIZATION DYNAMICS: We use a "Two-Speed" engine (Differential Learning Rates)
#    to ensure the model's pre-trained medical 'brain' isn't corrupted while the 
#    new classification 'mouth' learns at high speed.
# =========================================================================================

import subprocess, sys
# Ensure the Spanish NLP requirements are met before execution.
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import os
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score

# =========================================================================================
# 0. THE SEED LOCKER (Absolute Reproducibility)
# =========================================================================================
def seed_everything(seed=42):
    """
    Pinned seeds across all libraries to ensure that every weight initialization,
    data shuffle, and dropout mask is identical across different runs.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# =========================================================================================
# PART 1: THE DATA PARSER (Strategy B: Bi-Directional Context Expansion)
# =========================================================================================
def parse_grace_json(file_path):
    """
    Converts raw clinical text into a contextualized dataset. 
    It identifies labeled spans and fills in the gaps with background 'None' sentences.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    nlp = spacy.load("es_core_news_sm")
    records = []

    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]

        spacy_doc = nlp(raw_text)
        all_sents = list(spacy_doc.sents)
        sent_records = []  

        # Pass 1: Extract gold-standard argumentative components
        for ann in annotations:
            sent_records.append({"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]})

        # Pass 2: Extract non-argumentative background sentences (labeled as 'None')
        for sent in all_sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": s})

        # Re-establish chronological order for logical context extraction
        sent_records.sort(key=lambda x: x["start"])

        # Pass 3: Extract the "Context Window" (Previous + Next sentence)
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            
            # Combine surroundings into a single context block
            surrounding_context = f"{prev_text} [SEP] {next_text}".strip()
            
            records.append({
                "text": sr["text"], 
                "context": surrounding_context, 
                "label": sr["label"]
            })

    return pd.DataFrame(records)


# =========================================================================================
# PART 2: DATA LOADING & MODEL INITIALIZATION
# =========================================================================================
# Load Train and Dev sets using the new contextual parser
train_df = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_train.json")
dev_df   = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json")

# Map text labels to categorical integers for the loss function
label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}
train_df["label"] = train_df["label"].map(label_map)
dev_df["label"]   = dev_df["label"].map(label_map)

# Load the Spanish Clinical RoBERTa pre-trained on medical EHRs
MODEL_NAME = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nDownloading Clinical RoBERTa weights...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True 
)

def tokenize_fn(examples):
    """
    Encodes the target sentence against its surrounding context.
    We bump max_length to 256 to prevent clipping the bi-directional context.
    """
    return tokenizer(
        examples["context"], 
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=256
    )

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)


# =========================================================================================
# PART 3: EVALUATION INFRASTRUCTURE
# =========================================================================================
# Building the Dev template once to avoid expensive SpaCy re-runs during training.
print("Pre-building Dev set evaluation template...")
nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8"))

dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
    for sent in spacy_doc.sents:
        s, e = sent.start_char, sent.end_char
        if not any(max(s, a) < min(e, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

# --- SCORING FUNCTIONS (STRICT RULES) ---
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _enrich(entities, token_positions): return [{**e, "tokens": frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= e["start"] and te <= e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    return matched, unmatched_a, [items_b[j] for j in range(len(items_b)) if j not in used]

def evaluate_strict_f1(cases):
    """Calculates the official metric: Macro F1 under character-exact matching."""
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tokens = _tokenize(case["raw_text"])
        gold_ents = _enrich(case["annotations"]["entities"], tokens)
        pred_ents = _enrich(case["predictions"]["entities"], tokens)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        m, ug, up = _greedy_match(gold_comps, pred_comps, lambda a, b: a.type == b.type and a.start == b.start and a.end == b.end)
        for g, _ in m: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    f1s = []
    metrics = {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        tp, fp, fn = tp_s[t], fp_s[t], fn_s[t]
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        metrics[t] = f; f1s.append(f)
    metrics["Macro_F1"] = sum(f1s) / 3
    return metrics

def compute_metrics(eval_pred):
    """The epoch callback that scores the model using competition-style JSON reconstruction."""
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3  # MajorClaim recalibration
    text_to_pred = {text: pred for text, pred in zip(dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities = []
        for i, sr in enumerate(template["sent_records"]):
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{i}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    res = evaluate_strict_f1(cases)
    print(f"\n--- STRICT EPOCH SCORES ---\n🥇 Macro: {res['Macro_F1']:.4f} | 📊 Premise: {res['Premise']:.4f} | 📈 Claim: {res['Claim']:.4f} | 👑 Major: {res['MajorClaim']:.4f}\n")
    return {"strict_macro_f1": res["Macro_F1"]}


# =========================================================================================
# PART 4: TRAINING SETUP & THE TWO-SPEED OPTIMIZER
# =========================================================================================
counts  = np.bincount(train_df["label"].values, minlength=4).astype(float)
weights = ( (counts.sum() / (len(counts) * counts)) / (counts.sum() / (len(counts) * counts)).mean() ).tolist()
weights[3] = 10.0  # Force model to prioritize the critical but rare MajorClaim class

# --- DIFFERENTIAL LEARNING RATES ---
# Slow Base (1e-5) to preserve pre-trained knowledge.
# Fast Head (1e-4) to quickly map clinical logic.
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
base_params = [p for n, p in model.named_parameters() if "classifier" not in n]

custom_optimizer = torch.optim.AdamW([
    {"params": base_params, "lr": 1e-5},
    {"params": head_params, "lr": 1e-4}
], weight_decay=0.05) 

training_args = TrainingArguments(
    output_dir="./roberta_grace", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16, 
    per_device_eval_batch_size=32, num_train_epochs=15, fp16=True, 
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=42, data_seed=42
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights, dtype=outputs.logits.dtype, device=outputs.logits.device)
        loss = F.cross_entropy(outputs.logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedCETrainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=dev_dataset,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    optimizers=(custom_optimizer, None)
)

print("\nStarting Optimized Training...")
trainer.train()

# =========================================================================================
# PART 5: EXTRACTION
# =========================================================================================
print("\nExtracting Golden Predictions for Ensemble...")
logits = trainer.predict(dev_dataset).predictions
np.save("roberta_dev_logits.npy", logits)
print("✅ Golden Logits saved to roberta_dev_logits.npy")

In [5]:
# =========================================================================================
# 🧠 GRACE Track 1 — THE LOGIC EXPERT (DeBERTa-v3)
# Trains, saves logits to memory for ensembling, and generates its own submission JSON.
# =========================================================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import json
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score
from huggingface_hub import hf_hub_download

# ============================================================
# 1. Parsing
# ============================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    nlp     = spacy.load("es_core_news_sm")
    records = []

    for doc in data:
        raw_text    = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]

        spacy_doc  = nlp(raw_text)
        all_sents  = list(spacy_doc.sents)
        sent_records = []  

        for ann in annotations:
            sent_records.append({"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]})

        for sent in all_sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": s})

        sent_records.sort(key=lambda x: x["start"])

        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": sr["label"]})

    df = pd.DataFrame(records)
    # Do not drop duplicates to keep arrays aligned!
    return df

# ============================================================
# 2. Load & encode
# ============================================================
train_df = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json")
dev_df   = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json")

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
train_df["label"] = train_df["label"].map(label_map)
dev_df["label"]   = dev_df["label"].map(label_map)

# ============================================================
# 3. Load DeBERTa & Fix Dictionary
# ============================================================
MODEL_NAME = "microsoft/mdeberta-v3-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=4)
model  = AutoModelForSequenceClassification.from_config(config)

print("Downloading pre-trained weights …")
ckpt_path = hf_hub_download(repo_id=MODEL_NAME, filename="pytorch_model.bin")
raw_sd    = torch.load(ckpt_path, map_location="cpu")

remapped_sd = {
    k.replace("LayerNorm.gamma", "LayerNorm.weight").replace("LayerNorm.beta",  "LayerNorm.bias"): v
    for k, v in raw_sd.items()
}
model.load_state_dict(remapped_sd, strict=False)

def tokenize_fn(examples):
    return tokenizer(examples["prev_text"], examples["text"], padding="max_length", truncation=True, max_length=192)

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)

# ============================================================
# 4. Training Setup
# ===========================================================

counts  = np.bincount(train_df["label"].values, minlength=4).astype(float)
raw_w   = counts.sum() / (len(counts) * counts)
weights = (raw_w / raw_w.mean()).tolist()
weights[3] = 10.0 

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}

training_args = TrainingArguments(
    output_dir="./mdeberta_grace", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, warmup_steps=100, lr_scheduler_type="cosine",
    per_device_train_batch_size=16, per_device_eval_batch_size=32, num_train_epochs=15,           
    weight_decay=0.01, fp16=False, load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.1,
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        w       = torch.tensor(weights, dtype=logits.dtype, device=logits.device)
        loss    = F.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedCETrainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=dev_dataset,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\nStarting Training Loop...")
trainer.train()

# ============================================================
# 5. Save Logits & Create Submission File
# ============================================================
print("\nExtracting Predictions & Saving Memory for Ensemble...")
raw_preds = trainer.predict(dev_dataset)
logits = raw_preds.predictions
np.save("deberta_dev_logits.npy", logits)
print("✅ Saved deberta_dev_logits.npy!")

probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
standard_preds = np.argmax(probs, axis=-1)

final_preds = np.copy(standard_preds)
major_claim_mask = probs[:, 3] > 0.50
final_preds[major_claim_mask] = 3

text_to_pred = {}
for text, pred in zip(dev_df["text"], final_preds):
    text_to_pred[text] = pred

def create_submission_file(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    inv_label_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}
    submission = []
    
    print("Mapping predictions back to the original JSON...")
    for doc in data:
        doc_id = doc["id"]
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = []
        for ann in annotations:
            sent_records.append({"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]})
        for sent in spacy_doc.sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
        
        sent_records.sort(key=lambda x: x["start"])
        predicted_entities = []
        entity_counter = 1
        
        for sr in sent_records:
            pred_int = text_to_pred.get(sr["text"], 0)
            pred_label = inv_label_map[pred_int]
            if pred_label != "None":
                predicted_entities.append({"id": f"T{entity_counter}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": pred_label})
                entity_counter += 1
                
        submission.append({
            "id": doc_id, "raw_text": raw_text, "metadata": doc.get("metadata", {}),
            "annotations": {"entities": predicted_entities, "relations": []},
            "predictions": {"entities": predicted_entities, "relations": []}
        })
        
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=4)
    print(f"✅ Submission saved to {output_path}")

create_submission_file("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json", "deberta_dev_submission.json")

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Map:   0%|          | 0/4584 [00:00<?, ? examples/s]

Map:   0%|          | 0/705 [00:00<?, ? examples/s]


Starting Training Loop...


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.788611,0.641975
2,0.877208,0.823350,0.633424


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
# =========================================================================================
# 🧠 GRACE Track 1 — THE LOGIC EXPERT (DeBERTa-v3 Grandmaster Edition)
# 
# STRATEGY:
# 1. BI-DIRECTIONAL CONTEXT: Uses the [Prev Sentence] + [Target] + [Next Sentence] window.
# 2. SEED LOCKDOWN: Everything is pinned to Seed 42 for reproducibility.
# 3. DIFFERENTIAL OPTIMIZER: Slow base learning, fast head learning to prevent overfitting.
# 4. REAL-TIME STRICT EVAL: Scores each epoch using the official strict character-exact rules.
# =========================================================================================

import subprocess, sys
# 1. Download Spanish grammar engine
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import os
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score
from huggingface_hub import hf_hub_download

# =========================================================================================
# 0. LOCK THE UNIVERSE (Seed 42)
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# =========================================================================================
# PART 1: THE DATA PARSER (Strategy B: Bi-Directional Context Expansion)
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    nlp = spacy.load("es_core_news_sm")
    records = []

    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]

        spacy_doc = nlp(raw_text)
        all_sents = list(spacy_doc.sents)
        sent_records = []  

        for ann in annotations:
            sent_records.append({"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]})

        for sent in all_sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": s})

        sent_records.sort(key=lambda x: x["start"])

        # Bi-Directional Context Window
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            
            # Combine surroundings into context block
            context_string = f"{prev_text} [SEP] {next_text}".strip()
            
            records.append({
                "text": sr["text"], 
                "context": context_string, 
                "label": sr["label"]
            })

    return pd.DataFrame(records)

# =========================================================================================
# PART 2: LOAD DATA & PREPARE DEBERTA
# =========================================================================================
train_df = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json")
dev_df   = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json")

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

train_df["label"] = train_df["label"].map(label_map)
dev_df["label"]   = dev_df["label"].map(label_map)

MODEL_NAME = "microsoft/mdeberta-v3-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# DeBERTa LayerNorm Dictionary Hack
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=4)
model  = AutoModelForSequenceClassification.from_config(config)

print("Downloading DeBERTa-v3 pre-trained weights...")
ckpt_path = hf_hub_download(repo_id=MODEL_NAME, filename="pytorch_model.bin")
raw_sd    = torch.load(ckpt_path, map_location="cpu")
remapped_sd = {
    k.replace("LayerNorm.gamma", "LayerNorm.weight").replace("LayerNorm.beta", "LayerNorm.bias"): v
    for k, v in raw_sd.items()
}
model.load_state_dict(remapped_sd, strict=False)

def tokenize_fn(examples):
    return tokenizer(examples["context"], examples["text"], padding="max_length", truncation=True, max_length=256)

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)



# =========================================================================================
# PART 3: REAL-TIME SCORING ENGINE (STRICT)
# =========================================================================================
print("Pre-building Dev set evaluation template...")
nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json", "r", encoding="utf-8"))

dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
    for sent in spacy_doc.sents:
        s, e = sent.start_char, sent.end_char
        if not any(max(s, a) < min(e, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _enrich(entities, token_positions): return [{**e, "tokens": frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= e["start"] and te <= e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    return matched, unmatched_a, [items_b[j] for j in range(len(items_b)) if j not in used]

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tokens = _tokenize(case["raw_text"])
        gold_ents = _enrich(case["annotations"]["entities"], tokens)
        pred_ents = _enrich(case["predictions"]["entities"], tokens)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        m, ug, up = _greedy_match(gold_comps, pred_comps, lambda a, b: a.type == b.type and a.start == b.start and a.end == b.end)
        for g, _ in m: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    f1s = []
    metrics = {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        tp, fp, fn = tp_s[t], fp_s[t], fn_s[t]
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        metrics[t] = f; f1s.append(f)
    metrics["Macro_F1"] = sum(f1s) / 3
    return metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3  # Force logic check
    text_to_pred = {text: pred for text, pred in zip(dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities = []
        for i, sr in enumerate(template["sent_records"]):
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{i}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    res = evaluate_strict_f1(cases)
    print(f"\n--- DEBERTA STRICT EPOCH SCORES ---\n🥇 Macro: {res['Macro_F1']:.4f} | 📊 Premise: {res['Premise']:.4f} | 📈 Claim: {res['Claim']:.4f} | 👑 Major: {res['MajorClaim']:.4f}\n")
    return {"strict_macro_f1": res["Macro_F1"]}


# =========================================================================================
# PART 4: TRAINING SETUP & THE TWO-SPEED OPTIMIZER
# =========================================================================================
counts  = np.bincount(train_df["label"].values, minlength=4).astype(float)
weights = ( (counts.sum() / (len(counts) * counts)) / (counts.sum() / (len(counts) * counts)).mean() ).tolist()
weights[3] = 10.0  

# Differential Learning Rates: Base vs Head

head_params = [p for n, p in model.named_parameters() if "classifier" in n]

base_params = [p for n, p in model.named_parameters() if "classifier" not in n]

custom_optimizer = torch.optim.AdamW([
    {"params": base_params, "lr": 1e-5}, # Logic brain learns slowly
    {"params": head_params, "lr": 1e-4}  # Decision head learns fast
], weight_decay=0.05)

training_args = TrainingArguments(
    output_dir="./mdeberta_grace", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=12, 
    per_device_eval_batch_size=12, num_train_epochs=15, fp16=False, # Standard 32-bit for DeBERTa stability
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=42, data_seed=42
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights, dtype=outputs.logits.dtype, device=outputs.logits.device)
        loss = F.cross_entropy(outputs.logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedCETrainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=dev_dataset,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    optimizers=(custom_optimizer, None)
)

print("\nStarting Real-Time Evaluation Training Loop for Logic Expert...")
trainer.train()

# =========================================================================================
# PART 5: EXTRACTION
# =========================================================================================
print("\nExtracting Golden Predictions for Ensemble...")
logits = trainer.predict(dev_dataset).predictions
np.save("deberta_dev_logits.npy", logits)
print("✅ Golden Logits saved to deberta_dev_logits.npy (Reproducible with Seed 42)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 71.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Map:   0%|          | 0/4584 [00:00<?, ? examples/s]

Map:   0%|          | 0/705 [00:00<?, ? examples/s]

Pre-building Dev set evaluation template...

Starting Real-Time Evaluation Training Loop for Logic Expert...


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 736.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 565.81 MiB is free. Including non-PyTorch memory, this process has 14.01 GiB memory in use. Of the allocated memory 8.87 GiB is allocated by PyTorch, and 5.00 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# =========================================================================================
# 🧬 GRACE Track 1 — THE PURE CLINICAL EXPERT (bsc-bio-ehr-es)
# Grandmaster Edition: Real-Time Scoring + Seed Locked + Differential LR
# =========================================================================================

import subprocess, sys
# Ensure Spanish NLP engine is downloaded
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import os
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score

# =========================================================================================
# 0. THE SEED LOCKER (Absolute Reproducibility)
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# =========================================================================================
# PART 1: THE DATA PARSER (Strategy B: Bi-Directional Context Window)
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    nlp = spacy.load("es_core_news_sm")
    records = []

    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]

        spacy_doc = nlp(raw_text)
        all_sents = list(spacy_doc.sents)
        sent_records = []  

        for ann in annotations:
            sent_records.append({"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"], "end": ann["end"]})

        for sent in all_sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": s, "end": e})

        sent_records.sort(key=lambda x: x["start"])

        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            context_string = f"{prev_text} [SEP] {next_text}".strip()
            
            records.append({
                "text": sr["text"],
                "context": context_string,
                "label": sr["label"]
            })

    return pd.DataFrame(records)

# =========================================================================================
# PART 2: LOAD & ENCODE
# =========================================================================================
train_df = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_train.json")
dev_df   = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json")

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

train_df["label"] = train_df["label"].map(label_map)
dev_df["label"]   = dev_df["label"].map(label_map)

MODEL_NAME = "PlanTL-GOB-ES/bsc-bio-ehr-es"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"\nDownloading Pure Clinical weights for {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True 
)

def tokenize_fn(examples):
    return tokenizer(
        examples["context"], 
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=256
    )

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)

# =========================================================================================
# PART 3: IN-MEMORY SCORING CACHE
# =========================================================================================
print("Pre-building Dev set evaluation template...")
nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8"))

dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
    for sent in spacy_doc.sents:
        s, e = sent.start_char, sent.end_char
        if not any(max(s, a) < min(e, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _enrich(entities, tokens): return [{**e, "tokens": frozenset(i for i, (ts, te) in enumerate(tokens) if ts >= e["start"] and te <= e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    return matched, unmatched_a, [items_b[j] for j in range(len(items_b)) if j not in used]

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tokens = _tokenize(case["raw_text"])
        gold_ents = _enrich(case["annotations"]["entities"], tokens)
        pred_ents = _enrich(case["predictions"]["entities"], tokens)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        m, ug, up = _greedy_match(gold_comps, pred_comps, lambda a, b: a.type == b.type and a.start == b.start and a.end == b.end)
        for g, _ in m: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    f1s = []
    metrics = {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        tp, fp, fn = tp_s[t], fp_s[t], fn_s[t]
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        metrics[t] = f; f1s.append(f)
    metrics["Macro_F1"] = sum(f1s) / 3
    return metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3  # Recalibrate MajorClaims
    text_to_pred = {text: pred for text, pred in zip(dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities = []
        for i, sr in enumerate(template["sent_records"]):
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{i}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    res = evaluate_strict_f1(cases)
    print(f"\n--- BSC-BIO STRICT EPOCH SCORES ---\n🥇 Macro: {res['Macro_F1']:.4f} | 📊 Premise: {res['Premise']:.4f} | 📈 Claim: {res['Claim']:.4f} | 👑 Major: {res['MajorClaim']:.4f}\n")
    return {"strict_macro_f1": res["Macro_F1"]}

# =========================================================================================
# PART 4: TRAINING SETUP & THE TWO-SPEED OPTIMIZER
# =========================================================================================
counts  = np.bincount(train_df["label"].values, minlength=4).astype(float)
weights = ( (counts.sum() / (len(counts) * counts)) / (counts.sum() / (len(counts) * counts)).mean() ).tolist()
weights[3] = 10.0  

head_params = [p for n, p in model.named_parameters() if "classifier" in n]
base_params = [p for n, p in model.named_parameters() if "classifier" not in n]

custom_optimizer = torch.optim.AdamW([
    {"params": base_params, "lr": 1e-5}, # Brain
    {"params": head_params, "lr": 1e-4}  # Head
], weight_decay=0.05) 

training_args = TrainingArguments(
    output_dir="./bsc_bio_grace", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16, 
    per_device_eval_batch_size=32, num_train_epochs=15, fp16=True, 
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=42, data_seed=42
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights, dtype=outputs.logits.dtype, device=outputs.logits.device)
        loss = F.cross_entropy(outputs.logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedCETrainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=dev_dataset,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    optimizers=(custom_optimizer, None)
)

print("\nStarting Pure Clinical Training...")
trainer.train()

# =========================================================================================
# PART 5: EXTRACTION OF THE GOLDEN LOGITS
# =========================================================================================
print("\nExtracting Golden Predictions for Ensemble...")
logits = trainer.predict(dev_dataset).predictions
np.save("bsc_bio_dev_logits.npy", logits)
print("✅ Golden Logits saved to bsc_bio_dev_logits.npy (Reproducible with Seed 42)")

In [ ]:
# =========================================================================================
# ⚖️ THE WEIGHTED ENSEMBLE GRID SEARCH (DeBERTa + bsc-bio-ehr-es) [1,000 COMBINATIONS]
# =========================================================================================
import numpy as np
import json
import torch
import torch.nn.functional as F 
import spacy
import zipfile

print("Initiating bsc-bio + DeBERTa 1,000-Step Grid Search...")

# =========================================================================================
# STEP 1: LOAD THE NEW BRAINS
# =========================================================================================
try:
    deberta_logits = np.load("deberta_dev_logits.npy")
    bsc_logits = np.load("bsc_bio_dev_logits.npy") # <--- LOADING OUR NEW MEDICAL EXPERT
except FileNotFoundError:
    print("❌ ERROR: Cannot find the .npy files! Run the model training cells first.")
    raise

# Convert raw math into actual percentages (0.0 to 1.0)
deberta_probs = F.softmax(torch.tensor(deberta_logits), dim=-1).numpy()
bsc_probs = F.softmax(torch.tensor(bsc_logits), dim=-1).numpy()

# =========================================================================================
# STEP 2: PREPARE THE TEXT MAPPER
# =========================================================================================
def get_dev_texts():
    with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    texts = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10: 
                    sent_records.append({"text": sent.text.strip(), "start": s})
        sent_records.sort(key=lambda x: x["start"])
        for sr in sent_records: 
            texts.append(sr["text"])
    return texts

print("Parsing original Dev set sentences for mapping...")
dev_texts = get_dev_texts()
nlp = spacy.load("es_core_news_sm")
inv_label_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}

# =========================================================================================
# STEP 3: THE 1,000-STEP GRID SEARCH & ZIP LOOP
# =========================================================================================
# This generates exactly 1,000 steps from 0.001 to 0.999
deberta_weights = np.linspace(0.001, 0.999, 1000).tolist()
zip_filename = "ensemble_bsc_grid_1000.zip"

print(f"\nCreating archive: {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

    for i, w_deb in enumerate(deberta_weights):
        w_bsc = 1.0 - w_deb
        
        output_file = f"deb_{w_deb:.3f}_bsc_{w_bsc:.3f}.json"
        
        # --- THE WEIGHTED FUSION ---
        ensemble_probs = (w_deb * deberta_probs) + (w_bsc * bsc_probs)
        final_preds = np.argmax(ensemble_probs, axis=-1)
        
        # Apply a baseline 0.50 threshold for MajorClaims to help combat class imbalance
        final_preds[ensemble_probs[:, 3] > 0.50] = 3
        
        text_to_pred = {}
        for text, pred in zip(dev_texts, final_preds):
            text_to_pred[text] = pred
            
        with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
            data = json.load(f)
            
        submission = []
        
        for doc in data:
            doc_id = doc["id"]
            raw_text = doc["raw_text"]
            
            annotations = doc.get("annotations", {}).get("entities", [])
            annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
            spacy_doc = nlp(raw_text)
            
            sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
            for sent in spacy_doc.sents:
                s, e = sent.start_char, sent.end_char
                if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                    if len(sent.text.strip()) > 10: 
                        sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
            
            sent_records.sort(key=lambda x: x["start"])
            predicted_entities = []
            entity_counter = 1
            
            for sr in sent_records:
                pred_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
                if pred_label != "None":
                    predicted_entities.append({
                        "id": f"T{entity_counter}", 
                        "text": sr["text"], 
                        "start": sr["start"], 
                        "end": sr["end"], 
                        "type": pred_label
                    })
                    entity_counter += 1
                    
            submission.append({
                "id": doc_id, 
                "raw_text": raw_text, 
                "metadata": doc.get("metadata", {}),
                "annotations": {"entities": predicted_entities, "relations": []}, 
                "predictions": {"entities": predicted_entities, "relations": []}  
            })

        json_string = json.dumps(submission, ensure_ascii=False, indent=4)
        zipf.writestr(output_file, json_string)
            
        # Print progress every 100 iterations so your Kaggle console doesn't crash
        if i % 100 == 0:
            mc_count = np.sum(final_preds == 3)
            claim_count = np.sum(final_preds == 2)
            prem_count = np.sum(final_preds == 1)
            print(f"📦 Zipped {i}/1000 ({output_file}) -> {mc_count} MajorClaims | {claim_count} Claims | {prem_count} Premises")

print(f"\n🎉 Grid Search Complete! Download '{zip_filename}' and run the Smart Evaluator!")

In [ ]:
# =========================================================================================
# ⚖️ THE WEIGHTED ENSEMBLE GRID SEARCH (bsc-bio-ehr-es + Clinical RoBERTa) [1,000 COMBINATIONS]
# =========================================================================================
import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import zipfile

print("Initiating BSC-Bio + Clinical RoBERTa 1,000-Step Grid Search...")

# =========================================================================================
# STEP 1: LOAD THE BRAINS (The Two Medical Titans)
# =========================================================================================
try:
    bsc_logits = np.load("bsc_bio_dev_logits.npy")
    roberta_logits = np.load("roberta_dev_logits.npy") 
except FileNotFoundError:
    print("❌ ERROR: Cannot find the .npy files! Run the model training cells first.")
    raise

# Convert raw math into actual percentages (0.0 to 1.0)
bsc_probs = F.softmax(torch.tensor(bsc_logits), dim=-1).numpy()
roberta_probs = F.softmax(torch.tensor(roberta_logits), dim=-1).numpy()

# =========================================================================================
# STEP 2: PREPARE THE TEXT MAPPER
# =========================================================================================
def get_dev_texts():
    with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    texts = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10: 
                    sent_records.append({"text": sent.text.strip(), "start": s})
        sent_records.sort(key=lambda x: x["start"])
        for sr in sent_records: 
            texts.append(sr["text"])
    return texts

print("Parsing original Dev set sentences for mapping...")
dev_texts = get_dev_texts()
nlp = spacy.load("es_core_news_sm")
inv_label_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}

# =========================================================================================
# STEP 3: THE 1,000-STEP GRID SEARCH & ZIP LOOP
# =========================================================================================
# This generates exactly 1,000 steps from 0.001 to 0.999
bsc_weights = np.linspace(0.001, 0.999, 1000).tolist()
zip_filename = "ensemble_bsc_roberta_grid_1000.zip"

print(f"\nCreating archive: {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

    for i, w_bsc in enumerate(bsc_weights):
        w_rob = 1.0 - w_bsc
        
        output_file = f"bsc_{w_bsc:.3f}_rob_{w_rob:.3f}.json"
        
        # --- THE WEIGHTED FUSION ---
        ensemble_probs = (w_bsc * bsc_probs) + (w_rob * roberta_probs)
        final_preds = np.argmax(ensemble_probs, axis=-1)
        
        # Apply a baseline 0.50 threshold for MajorClaims to help combat class imbalance
        final_preds[ensemble_probs[:, 3] > 0.50] = 3
        
        text_to_pred = {}
        for text, pred in zip(dev_texts, final_preds):
            text_to_pred[text] = pred
            
        with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
            data = json.load(f)
            
        submission = []
        
        for doc in data:
            doc_id = doc["id"]
            raw_text = doc["raw_text"]
            
            annotations = doc.get("annotations", {}).get("entities", [])
            annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
            spacy_doc = nlp(raw_text)
            
            sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
            for sent in spacy_doc.sents:
                s, e = sent.start_char, sent.end_char
                if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                    if len(sent.text.strip()) > 10: 
                        sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
            
            sent_records.sort(key=lambda x: x["start"])
            predicted_entities = []
            entity_counter = 1
            
            for sr in sent_records:
                pred_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
                if pred_label != "None":
                    predicted_entities.append({
                        "id": f"T{entity_counter}", 
                        "text": sr["text"], 
                        "start": sr["start"], 
                        "end": sr["end"], 
                        "type": pred_label
                    })
                    entity_counter += 1
                    
            submission.append({
                "id": doc_id, 
                "raw_text": raw_text, 
                "metadata": doc.get("metadata", {}),
                "annotations": {"entities": predicted_entities, "relations": []}, 
                "predictions": {"entities": predicted_entities, "relations": []}  
            })

        json_string = json.dumps(submission, ensure_ascii=False, indent=4)
        zipf.writestr(output_file, json_string)
            
        # Print progress every 100 iterations so your Kaggle console doesn't crash
        if i % 100 == 0:
            mc_count = np.sum(final_preds == 3)
            claim_count = np.sum(final_preds == 2)
            prem_count = np.sum(final_preds == 1)
            print(f"📦 Zipped {i}/1000 ({output_file}) -> {mc_count} MajorClaims | {claim_count} Claims | {prem_count} Premises")

print(f"\n🎉 Grid Search Complete! Download '{zip_filename}' and run the Smart Evaluator!")

In [ ]:
# =========================================================================================
# 👑 THE HOLY TRINITY ENSEMBLE: 3-WAY GRID SEARCH
# Models: DeBERTa (Logic) + Clinical RoBERTa (Hybrid) + BSC-Bio-EHR (Pure Medical)
# =========================================================================================
import subprocess, sys
# Auto-download the Spanish grammar engine so it doesn't crash on fresh sessions
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import zipfile

print("Initiating 3-Way Tri-Model Grid Search...")

# =========================================================================================
# STEP 1: LOAD ALL THREE BRAINS
# =========================================================================================
try:
    deberta_logits = np.load("/kaggle/input/datasets/alexcristea72/3-logits/deberta_dev_logits.npy")
    roberta_logits = np.load("/kaggle/input/datasets/alexcristea72/3-logits/roberta_dev_logits.npy")
    bsc_logits     = np.load("/kaggle/input/datasets/alexcristea72/3-logits/bsc_bio_dev_logits.npy")
except FileNotFoundError:
    print("❌ ERROR: Cannot find all three .npy files! Run the model training cells first.")
    raise

# Convert raw math into actual percentages (0.0 to 1.0)
deberta_probs = F.softmax(torch.tensor(deberta_logits), dim=-1).numpy()
roberta_probs = F.softmax(torch.tensor(roberta_logits), dim=-1).numpy()
bsc_probs     = F.softmax(torch.tensor(bsc_logits), dim=-1).numpy()

# =========================================================================================
# STEP 2: PREPARE THE TEXT MAPPER
# =========================================================================================
def get_dev_texts():
    with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    texts = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10: 
                    sent_records.append({"text": sent.text.strip(), "start": s})
        sent_records.sort(key=lambda x: x["start"])
        for sr in sent_records: 
            texts.append(sr["text"])
    return texts

print("Parsing original Dev set sentences for mapping...")
dev_texts = get_dev_texts()
nlp = spacy.load("es_core_news_sm")
inv_label_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}

# =========================================================================================
# STEP 3: THE 3-WAY GRID SEARCH & ZIP LOOP
# =========================================================================================
# We step by 0.05 (5%) to keep the combinatorial explosion manageable (generates 231 files).
steps = np.arange(0.0, 1.05, 0.05)
zip_filename = "ensemble_holy_trinity_grid.zip"

print(f"\nCreating archive: {zip_filename}...")

file_count = 0

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

    # Loop 1: DeBERTa
    for w_deb in steps:
        # Loop 2: Clinical RoBERTa
        for w_rob in steps:
            # The remaining percentage goes to BSC-Bio
            w_bsc = 1.0 - w_deb - w_rob
            
            # Skip invalid combinations (e.g., 80% DeBERTa + 80% RoBERTa = -60% BSC)
            if w_bsc < -0.001:
                continue
                
            # Clean up tiny Python floating point errors (e.g., 0.0000000000001)
            w_deb = max(0.0, min(1.0, w_deb))
            w_rob = max(0.0, min(1.0, w_rob))
            w_bsc = max(0.0, min(1.0, w_bsc))
            
            # Format filename to look like: deb_40_rob_35_bsc_25.json
            pct_deb = int(round(w_deb * 100))
            pct_rob = int(round(w_rob * 100))
            pct_bsc = int(round(w_bsc * 100))
            
            output_file = f"deb_{pct_deb:02d}_rob_{pct_rob:02d}_bsc_{pct_bsc:02d}.json"
            
            # --- THE 3-WAY WEIGHTED FUSION ---
            ensemble_probs = (w_deb * deberta_probs) + (w_rob * roberta_probs) + (w_bsc * bsc_probs)
            final_preds = np.argmax(ensemble_probs, axis=-1)
            
            # Apply a baseline 0.50 threshold for MajorClaims to help combat class imbalance
            final_preds[ensemble_probs[:, 3] > 0.50] = 3
            
            text_to_pred = {}
            for text, pred in zip(dev_texts, final_preds):
                text_to_pred[text] = pred
                
            with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
                data = json.load(f)
                
            submission = []
            
            for doc in data:
                doc_id = doc["id"]
                raw_text = doc["raw_text"]
                
                annotations = doc.get("annotations", {}).get("entities", [])
                annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
                spacy_doc = nlp(raw_text)
                
                sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
                for sent in spacy_doc.sents:
                    s, e = sent.start_char, sent.end_char
                    if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                        if len(sent.text.strip()) > 10: 
                            sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
                
                sent_records.sort(key=lambda x: x["start"])
                predicted_entities = []
                entity_counter = 1
                
                for sr in sent_records:
                    pred_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
                    if pred_label != "None":
                        predicted_entities.append({
                            "id": f"T{entity_counter}", 
                            "text": sr["text"], 
                            "start": sr["start"], 
                            "end": sr["end"], 
                            "type": pred_label
                        })
                        entity_counter += 1
                        
                submission.append({
                    "id": doc_id, 
                    "raw_text": raw_text, 
                    "metadata": doc.get("metadata", {}),
                    "annotations": {"entities": predicted_entities, "relations": []}, 
                    "predictions": {"entities": predicted_entities, "relations": []}  
                })

            json_string = json.dumps(submission, ensure_ascii=False, indent=4)
            zipf.writestr(output_file, json_string)
            file_count += 1
                
            # Print progress every 50 iterations
            if file_count % 50 == 0:
                mc_count = np.sum(final_preds == 3)
                claim_count = np.sum(final_preds == 2)
                prem_count = np.sum(final_preds == 1)
                print(f"📦 Zipped {file_count}/231 ({output_file}) -> {mc_count} MajorClaims | {claim_count} Claims | {prem_count} Premises")

print(f"\n🎉 3-Way Grid Search Complete! Safely packed {file_count} files into '{zip_filename}'.")

In [ ]:
# =========================================================================================
# ⚖️ THE WEIGHTED ENSEMBLE GRID SEARCH (ZIP EDITION)
# Goal: Test every weight ratio between DeBERTa and RoBERTa, and package all 
# submission JSON files neatly into a single, easily downloadable ZIP archive.
# =========================================================================================
import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import zipfile # <--- The magic library to keep our Kaggle directory clean

print("Initiating Weighted Ensemble Grid Search...")

# =========================================================================================
# STEP 1: LOAD THE BRAINS
# =========================================================================================
try:
    deberta_logits = np.load("deberta_dev_logits.npy")
    roberta_logits = np.load("roberta_dev_logits.npy")
except FileNotFoundError:
    print("❌ ERROR: Cannot find the .npy files! Run the model training cells first.")
    raise

# Convert raw math into actual percentages (0.0 to 1.0)
deberta_probs = F.softmax(torch.tensor(deberta_logits), dim=-1).numpy()
roberta_probs = F.softmax(torch.tensor(roberta_logits), dim=-1).numpy()

# =========================================================================================
# STEP 2: PREPARE THE TEXT MAPPER
# We extract the texts once so we don't have to do it 9 times in a loop.
# =========================================================================================
def get_dev_texts():
    with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    texts = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10: 
                    sent_records.append({"text": sent.text.strip(), "start": s})
        sent_records.sort(key=lambda x: x["start"])
        for sr in sent_records: 
            texts.append(sr["text"])
    return texts

print("Parsing original Dev set sentences for mapping...")
dev_texts = get_dev_texts()
nlp = spacy.load("es_core_news_sm")
inv_label_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}

# =========================================================================================
# STEP 3: THE GRID SEARCH & ZIP LOOP
# We will test DeBERTa weights from 10% (0.1) up to 90% (0.9)
# =========================================================================================
deberta_weights = np.linspace(0.001, 0.999, 1000).tolist()
zip_filename = "ensemble_submissions_grid.zip"

print(f"\nCreating archive: {zip_filename}...")

# Open a ZipFile in write mode
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

    for w_deb in deberta_weights:
        w_rob = 1.0 - w_deb
        
        # Format the percentages for the filename (e.g., 40 and 60)
        pct_deb = int(round(w_deb * 100))
        pct_rob = int(round(w_rob * 100))
        output_file = f"deb_{w_deb:.3f}_rob_{w_rob:.3f}.json"
        
        # --- THE WEIGHTED FUSION ---
        # Multiply each model's confidence by its assigned weight, then add them together
        ensemble_probs = (w_deb * deberta_probs) + (w_rob * roberta_probs)
        
        # Pick the class with the highest weighted average
        final_preds = np.argmax(ensemble_probs, axis=-1)
        
        # Apply a baseline 0.50 threshold for MajorClaims to help combat the severe class imbalance
        final_preds[ensemble_probs[:, 3] > 0.50] = 3
        
        # --- THE DICTIONARY HACK ---
        text_to_pred = {}
        for text, pred in zip(dev_texts, final_preds):
            text_to_pred[text] = pred
            
        # --- BUILD THE JSON FOR THIS SPECIFIC BLEND ---
        with open("/kaggle/input/datasets/alexcristea72/iberlef-grace/public_data/track_1_dev.json", "r", encoding="utf-8") as f:
            data = json.load(f)
            
        submission = []
        
        for doc in data:
            doc_id = doc["id"]
            raw_text = doc["raw_text"]
            
            annotations = doc.get("annotations", {}).get("entities", [])
            annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
            spacy_doc = nlp(raw_text)
            
            sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
            for sent in spacy_doc.sents:
                s, e = sent.start_char, sent.end_char
                if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                    if len(sent.text.strip()) > 10: 
                        sent_records.append({"text": sent.text.strip(), "start": s, "end": e})
            
            sent_records.sort(key=lambda x: x["start"])
            predicted_entities = []
            entity_counter = 1
            
            for sr in sent_records:
                pred_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
                if pred_label != "None":
                    predicted_entities.append({
                        "id": f"T{entity_counter}", 
                        "text": sr["text"], 
                        "start": sr["start"], 
                        "end": sr["end"], 
                        "type": pred_label
                    })
                    entity_counter += 1
                    
            submission.append({
                "id": doc_id, 
                "raw_text": raw_text, 
                "metadata": doc.get("metadata", {}),
                "annotations": {"entities": predicted_entities, "relations": []}, # Evaluator appease
                "predictions": {"entities": predicted_entities, "relations": []}  # Evaluator appease
            })

        # --- INJECT DIRECTLY INTO THE ZIP FILE ---
        # Convert the Python dictionary into a formatted JSON string
        json_string = json.dumps(submission, ensure_ascii=False, indent=4)
        # Write that string directly into the zip archive
        zipf.writestr(output_file, json_string)
            
        # Print out a quick summary so you can see how aggressive the model is being
        mc_count = np.sum(final_preds == 3)
        claim_count = np.sum(final_preds == 2)
        prem_count = np.sum(final_preds == 1)
        print(f"📦 Zipped {output_file} -> {mc_count} MajorClaims | {claim_count} Claims | {prem_count} Premises")

print(f"\n🎉 Grid Search Complete! All 9 JSON files are safely packed inside '{zip_filename}'.")

In [5]:
# =========================================================================================
# GRACE Track 1 — MrBERT Biomed Expert (Real-Time Scoring + Seed Locked)
# =========================================================================================

import os

# Force single GPU before importing torch/transformers.
# This avoids ModernBERT/MrBERT crashing under PyTorch DataParallel on multi-GPU Kaggle.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys

# 1. Download Spanish grammar engine silently
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score

# =========================================================================================
# 0. LOCK THE UNIVERSE (Seed Everything)
# =========================================================================================

def seed_everything(seed=21):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)

# =========================================================================================
# PART 1: THE DATA PARSER (Prev-Sentence Context Included)
# =========================================================================================

def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    nlp = spacy.load("es_core_news_sm")
    records = []

    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]

        spacy_doc = nlp(raw_text)
        all_sents = list(spacy_doc.sents)
        sent_records = []

        for ann in annotations:
            sent_records.append({
                "text": ann["text"].strip(),
                "label": ann["type"],
                "start": ann["start"]
            })

        for sent in all_sents:
            s, e = sent.start_char, sent.end_char

            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({
                        "text": sent.text.strip(),
                        "label": "None",
                        "start": s
                    })

        sent_records.sort(key=lambda x: x["start"])

        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({
                "text": sr["text"],
                "prev_text": prev_text,
                "label": sr["label"]
            })

    return pd.DataFrame(records)


# =========================================================================================
# PART 2: LOAD & ENCODE
# =========================================================================================

train_df = parse_grace_json(
    "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json"
)

dev_df = parse_grace_json(
    "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
)

label_map = {
    "None": 0,
    "Premise": 1,
    "Claim": 2,
    "MajorClaim": 3
}

inv_label_map = {v: k for k, v in label_map.items()}

train_df["label"] = train_df["label"].map(label_map)
dev_df["label"] = dev_df["label"].map(label_map)

MODEL_NAME = "BSC-LT/MrBERT-biomed"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nDownloading MrBERT Biomed weights...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    ignore_mismatched_sizes=True,
    attn_implementation="eager"
)

def tokenize_fn(examples):
    return tokenizer(
        examples["prev_text"],
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=192,
        return_token_type_ids=False
    )


train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)

# =========================================================================================
# PART 3: IN-MEMORY SCORING CACHE (Optimization)
# =========================================================================================

print("Pre-building Dev set evaluation template...")

nlp_cache = spacy.load("es_core_news_sm")

dev_raw_data = json.load(open(
    "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json",
    "r",
    encoding="utf-8"
))

dev_eval_template = []

for doc in dev_raw_data:
    raw_text = doc["raw_text"]
    annotations = doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    spacy_doc = nlp_cache(raw_text)

    sent_records = [
        {
            "text": ann["text"].strip(),
            "start": ann["start"],
            "end": ann["end"]
        }
        for ann in annotations
    ]

    for sent in spacy_doc.sents:
        s, e = sent.start_char, sent.end_char

        if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
            if len(sent.text.strip()) > 10:
                sent_records.append({
                    "text": sent.text.strip(),
                    "start": s,
                    "end": e
                })

    sent_records.sort(key=lambda x: x["start"])

    dev_eval_template.append({
        "id": doc["id"],
        "raw_text": raw_text,
        "annotations": {"entities": annotations},
        "sent_records": sent_records
    })


# =========================================================================================
# PART 4: THE OFFICIAL SCORING ENGINE (Stripped down to Strict rules)
# =========================================================================================

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def _tokenize(text: str):
    return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


def _span_token_set(token_positions, start, end):
    return frozenset(
        i for i, (ts, te) in enumerate(token_positions)
        if ts >= start and te <= end
    )


def _enrich(entities, token_positions):
    return [
        {
            **e,
            "tokens": _span_token_set(token_positions, e["start"], e["end"])
        }
        for e in entities
    ]


class CompTuple(NamedTuple):
    start: int
    end: int
    type: str
    tokens: frozenset


def _exact_span(a, b):
    return a.type == b.type and a.start == b.start and a.end == b.end


def _greedy_match(items_a, items_b, match_fn):
    used = set()
    matched = []
    unmatched_a = []

    for a in items_a:
        j = next(
            (j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)),
            None
        )

        if j is not None:
            matched.append((a, items_b[j]))
            used.add(j)
        else:
            unmatched_a.append(a)

    unmatched_b = [
        items_b[j]
        for j in range(len(items_b))
        if j not in used
    ]

    return matched, unmatched_a, unmatched_b


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return f


def evaluate_strict_f1(cases):
    tp_s = defaultdict(int)
    fp_s = defaultdict(int)
    fn_s = defaultdict(int)

    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))

        gold_ents = _enrich(case["annotations"]["entities"], tp)
        pred_ents = _enrich(case["predictions"]["entities"], tp)

        gold_comps = [
            CompTuple(e["start"], e["end"], e["type"], e["tokens"])
            for e in gold_ents
        ]

        pred_comps = [
            CompTuple(e["start"], e["end"], e["type"], e["tokens"])
            for e in pred_ents
        ]

        matched, unmatched_gold, unmatched_pred = _greedy_match(
            gold_comps,
            pred_comps,
            _exact_span
        )

        for g, _ in matched:
            tp_s[g.type] += 1

        for r in unmatched_pred:
            fp_s[r.type] += 1

        for r in unmatched_gold:
            fn_s[r.type] += 1

    f1s = []
    metrics = {}

    for t in ["Premise", "Claim", "MajorClaim"]:
        f = _prf(tp_s[t], fp_s[t], fn_s[t])
        metrics[t] = f
        f1s.append(f)

    metrics["Macro_F1"] = sum(f1s) / len(f1s) if f1s else 0.0

    return metrics


# =========================================================================================
# PART 5: THE LIVE EPOCH CALLBACK
# =========================================================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)

    # 0.50 MajorClaim override applied in memory
    preds[probs[:, 3] > 0.50] = 3

    text_to_pred = {
        text: pred
        for text, pred in zip(dev_df["text"], preds)
    }

    cases = []

    
    for template in dev_eval_template:
        predicted_entities = []
        e_id = 1

        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")

            if p_label != "None":
                predicted_entities.append({
                    "id": f"T{e_id}",
                    "text": sr["text"],
                    "start": sr["start"],
                    "end": sr["end"],
                    "type": p_label
                })
                e_id += 1

        cases.append({
            "raw_text": template["raw_text"],
            "annotations": template["annotations"],
            "predictions": {"entities": predicted_entities}
        })

    metrics = evaluate_strict_f1(cases)

    print("\n--- EPOCH EVALUATION ---")
    print(f"Macro F1   : {metrics['Macro_F1']:.4f}")
    print(f"Premise    : {metrics['Premise']:.4f}")
    print(f"Claim      : {metrics['Claim']:.4f}")
    print(f"MajorClaim : {metrics['MajorClaim']:.4f}")
    print("------------------------\n")

    return {
        "strict_macro_f1": metrics["Macro_F1"]
    }


# =========================================================================================
# PART 6: TRAINING SETUP & DIFFERENTIAL OPTIMIZER
# =========================================================================================

counts = np.bincount(train_df["label"].values, minlength=4).astype(float)
raw_w = counts.sum() / (len(counts) * counts)
weights = (raw_w / raw_w.mean()).tolist()

# Extra weight for rare MajorClaim class
weights[3] = 10.0

# Split weights: slow learning for the pretrained body, fast learning for the new head
head_params = [
    p for n, p in model.named_parameters()
    if "classifier" in n
]

base_params = [
    p for n, p in model.named_parameters()
    if "classifier" not in n
]

custom_optimizer = torch.optim.AdamW(
    [
        {"params": base_params, "lr": 1e-5},
        {"params": head_params, "lr": 1e-4}
    ],
    weight_decay=0.05
)

training_args = TrainingArguments(
    output_dir="./mrbert_biomed_grace",
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=100,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="strict_macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    save_only_model=True,
    report_to="none",
    label_smoothing_factor=0.15,
    seed=42,
    data_seed=42
)


class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        w = torch.tensor(
            weights,
            dtype=logits.dtype,
            device=logits.device
        )

        loss = F.cross_entropy(
            logits,
            labels,
            weight=w
        )

        return (loss, outputs) if return_outputs else loss


trainer = WeightedCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=4)
    ],
    optimizers=(custom_optimizer, None)
)

print("\nStarting Real-Time Evaluation Training Loop...")

trainer.train()

# =========================================================================================
# PART 7: EXTRACTION OF THE GOLDEN LOGITS
# =========================================================================================

print("\nExtracting Golden Predictions for Ensemble...")

raw_preds = trainer.predict(dev_dataset)
logits = raw_preds.predictions

np.save("mrbert_biomed_dev_logits.npy", logits)
print("Saved mrbert_biomed_dev_logits.npy (Best Epoch Captured)!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 81.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: BSC-LT/MrBERT-biomed
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/4584 [00:00<?, ? examples/s]

Map:   0%|          | 0/705 [00:00<?, ? examples/s]

Pre-building Dev set evaluation template...

Starting Real-Time Evaluation Training Loop...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [1]:
# =========================================================================================
# ⚡ THE LIGHTNING QUADRINITY: ULTRA-FAST 4-WAY GRID SEARCH
# Models: MrBERT + DeBERTa + RoBERTa + BSC-Bio
# =========================================================================================
import subprocess, sys

# 1. Download Spanish grammar engine
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=True)

import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import zipfile
import os
import time

print("⚡ Initiating Ultra-Fast 4-Way Grid Search...")

# 2. DEFINE PATHS
LOGIT_DIR = "/kaggle/input/datasets/alexcristea72/3-logits/"
DEV_JSON_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"

# 3. LOAD ALL FOUR BRAINS
try:
    p_mr  = F.softmax(torch.tensor(np.load(os.path.join(LOGIT_DIR, "mrbert_biomed_dev_logits.npy"))), dim=-1).numpy()
    p_deb = F.softmax(torch.tensor(np.load(os.path.join(LOGIT_DIR, "deberta_dev_logits.npy"))), dim=-1).numpy()
    p_rob = F.softmax(torch.tensor(np.load(os.path.join(LOGIT_DIR, "roberta_dev_logits.npy"))), dim=-1).numpy()
    p_bsc = F.softmax(torch.tensor(np.load(os.path.join(LOGIT_DIR, "bsc_bio_dev_logits.npy"))), dim=-1).numpy()
    print("✅ All 4 logit files loaded successfully.")
except FileNotFoundError as e:
    print(f"❌ ERROR: Could not find files in {LOGIT_DIR}. check your dataset names!")
    raise

# =========================================================================================
# 4. PRE-BUILD THE SUBMISSION SKELETON (DO THIS ONLY ONCE!)
# This skips running SpaCy 286 times and does it exactly 1 time.
# =========================================================================================
print("⏳ Pre-computing the Dev Set Template (This takes a few seconds, but saves hours)...")
with open(DEV_JSON_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

nlp = spacy.load("es_core_news_sm")
submission_skeleton = []
all_sentences = [] # Keep a flat list to map our flat numpy array of predictions

for doc in dev_data:
    raw_text = doc["raw_text"]
    annotations = doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    
    spacy_doc = nlp(raw_text)
    sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
    
    for sent in spacy_doc.sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
            if len(sent.text.strip()) > 10: 
                sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
                
    sent_records.sort(key=lambda x: x["start"])
    
    # Store the exact order of sentences to match our logits array
    for sr in sent_records:
        all_sentences.append(sr["text"])
        
    submission_skeleton.append({
        "id": doc["id"],
        "raw_text": raw_text,
        "metadata": doc.get("metadata", {}),
        "annotations": {"entities": annotations, "relations": []},
        "sent_records_template": sent_records # We will use this to quickly build predictions
    })

inv_label_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}
print("✅ Skeleton built! Starting the lightning loop...\n")


# =========================================================================================
# 5. THE ULTRA-FAST 4-WAY GRID SEARCH
# =========================================================================================
steps = np.arange(0.0, 1.1, 0.1)
zip_filename = "ensemble_quadrinity_results_fast.zip"
file_count = 0

start_time = time.time()

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for w_mr in steps:
        for w_deb in steps:
            for w_rob in steps:
                w_bsc = 1.0 - w_mr - w_deb - w_rob
                if w_bsc < -0.001: continue
                
                w_mr, w_deb, w_rob, w_bsc = [max(0.0, min(1.0, w)) for w in [w_mr, w_deb, w_rob, w_bsc]]
                fname = f"mr_{int(round(w_mr*100))}_deb_{int(round(w_deb*100))}_rob_{int(round(w_rob*100))}_bsc_{int(round(w_bsc*100))}.json"
                
                # 1. NumPy Vectorized Math (Instantly calculates all predictions)
                ensemble_probs = (w_mr * p_mr) + (w_deb * p_deb) + (w_rob * p_rob) + (w_bsc * p_bsc)
                final_preds = np.argmax(ensemble_probs, axis=-1)
                final_preds[ensemble_probs[:, 3] > 0.50] = 3 
                
                # Create a fast dictionary for O(1) lookups
                text_to_pred = {text: pred for text, pred in zip(all_sentences, final_preds)}
                
                # 2. Build Submission JSON directly from the Skeleton
                submission = []
                for skel in submission_skeleton:
                    ents = []
                    for i, sr in enumerate(skel["sent_records_template"]):
                        lbl = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
                        if lbl != "None": 
                            ents.append({"id": f"T{i+1}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": lbl})
                    
                    submission.append({
                        "id": skel["id"], 
                        "raw_text": skel["raw_text"], 
                        "metadata": skel["metadata"],
                        "annotations": skel["annotations"], 
                        "predictions": {"entities": ents, "relations": []}
                    })

                # 3. Zip it immediately
                zipf.writestr(fname, json.dumps(submission, ensure_ascii=False, indent=4))
                file_count += 1
                
                # Print progress to confirm the speed boost
                if file_count % 50 == 0: 
                    elapsed = time.time() - start_time
                    print(f"📦 [{file_count}/286] Zipped {fname} | Elapsed time: {elapsed:.2f}s")

total_time = time.time() - start_time
print(f"\n🎉 BOOM! Grid Search Complete in {total_time:.2f} seconds!")
print(f"Packed {file_count} files into '{zip_filename}'.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 83.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
⚡ Initiating Ultra-Fast 4-Way Grid Search...
✅ All 4 logit files loaded successfully.
⏳ Pre-computing the Dev Set Template (This takes a few seconds, but saves hours)...
✅ Skeleton built! Starting the lightning loop...

📦 [50/286] Zipped mr_0_deb_50_rob_40_bsc_10.json | Elapsed time: 0.81s
📦 [100/286] Zipped mr_10_deb_30_rob_60_bsc_0.json | Elapsed time: 1.62s
📦 [150/286] Zipped mr_20_deb_30_rob_40_bsc_10.json | Elapsed time: 2.40s
📦 [200/286] Zipped mr_30_deb_60_rob_0_bsc_10.json | Elapsed time: 3.18s
📦 [250/286] Zipped mr_50_deb_40_rob_10_bsc_0.json | Elapsed time: 3.99s

🎉

In [2]:
# =========================================================================================
# 🏆 GRACE Track 1 — THE FINAL SUBMISSION (MrBERT Biomed Full-Fit)
# Trains on Train + Dev. Predicts on Blind Test.
# =========================================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=True)

import random
import json
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# =========================================================================================
# 0. LOCK THE UNIVERSE
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# =========================================================================================
# PART 1: THE PARSERS (Training vs. Blind Inference)
# =========================================================================================
# Standard Parser for Training Data (Needs annotations)
def parse_training_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]} for ann in annotations]
        
        for sent in spacy_doc.sents:
            if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": sent.start_char})
                    
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": sr["label"]})
    return pd.DataFrame(records)

# NEW: Blind Parser (No annotations exist here)
def parse_blind_test(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    skeleton = [] # Keep track of document structure for JSON rebuilding
    
    for doc in data:
        raw_text = doc["raw_text"]
        spacy_doc = nlp(raw_text)
        sent_records = []
        
        for sent in spacy_doc.sents:
            if len(sent.text.strip()) > 10:
                sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
                
        sent_records.sort(key=lambda x: x["start"])
        
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            # Label is 0 (None) just as a placeholder for the Dataset object
            records.append({"text": sr["text"], "prev_text": prev_text, "label": 0}) 
            
        skeleton.append({
            "id": doc["id"], "raw_text": raw_text, "metadata": doc.get("metadata", {}),
            "sent_records": sent_records
        })
        
    return pd.DataFrame(records), skeleton

# =========================================================================================
# PART 2: COMBINE DATA & ENCODE
# =========================================================================================
print("Loading and combining Train + Dev sets for maximum learning...")
train_df = parse_training_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json")
dev_df   = parse_training_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json")

# COMBINE THEM!
full_train_df = pd.concat([train_df, dev_df], ignore_axis=True)

test_df, test_skeleton = parse_blind_test("/kaggle/input/datasets/alexcristea72/iberlef-grace/GRACE_2026_blind_test/track_1_blind_test.json")

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}
full_train_df["label"] = full_train_df["label"].map(label_map)

MODEL_NAME = "BSC-LT/MrBERT-biomed"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)

def tokenize_fn(examples):
    return tokenizer(examples["prev_text"], examples["text"], padding="max_length", truncation=True, max_length=192)

full_train_dataset = Dataset.from_pandas(full_train_df).map(tokenize_fn, batched=True)
test_dataset       = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True)

# =========================================================================================
# PART 3: THE "BLIND" TRAINING SETUP
# No Eval set. No Early Stopping. Just pure learning for 5 epochs.
# =========================================================================================
counts = np.bincount(full_train_df["label"].values, minlength=4).astype(float)
weights = ((counts.sum() / (len(counts) * counts)) / (counts.sum() / (len(counts) * counts)).mean()).tolist()
weights[3] = 10.0 # Keep MajorClaim boost

head_params = [p for n, p in model.named_parameters() if "classifier" in n]
base_params = [p for n, p in model.named_parameters() if "classifier" not in n]
custom_optimizer = torch.optim.AdamW([{"params": base_params, "lr": 1e-5}, {"params": head_params, "lr": 1e-4}], weight_decay=0.05)

# Hardcoded to 5 epochs (Adjust based on what epoch performed best during your Dev runs)
training_args = TrainingArguments(
    output_dir="./mrbert_final",
    warmup_steps=100, lr_scheduler_type="cosine",
    per_device_train_batch_size=16, 
    num_train_epochs=5, # <-- FIXED EPOCHS 
    fp16=True, 
    save_strategy="no", # Save time, don't save checkpoints
    report_to="none", label_smoothing_factor=0.15,
    seed=42, data_seed=42
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights, dtype=outputs.logits.dtype, device=outputs.logits.device)
        loss = F.cross_entropy(outputs.logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedCETrainer(model=model, args=training_args, train_dataset=full_train_dataset, optimizers=(custom_optimizer, None))

print("\n🚀 Starting Full-Fit Training on Train+Dev (No Validation)...")
trainer.train()

# =========================================================================================
# PART 4: INFERENCE & SUBMISSION GENERATION
# =========================================================================================
print("\n🔮 Predicting on Blind Test Set...")
test_logits = trainer.predict(test_dataset).predictions
np.save("mrbert_blind_test_logits.npy", test_logits) # Save logits just in case you want to ensemble later!

probs = F.softmax(torch.tensor(test_logits), dim=-1).numpy()
preds = np.argmax(probs, axis=-1)
preds[probs[:, 3] > 0.50] = 3 # MajorClaim threshold

text_to_pred = {text: pred for text, pred in zip(test_df["text"], preds)}

print("🏗️ Rebuilding final submission JSON...")
submission = []
for doc in test_skeleton:
    predicted_entities = []
    e_id = 1
    for sr in doc["sent_records"]:
        p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
        if p_label != "None":
            predicted_entities.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
            e_id += 1
            
    submission.append({
        "id": doc["id"], "raw_text": doc["raw_text"], "metadata": doc["metadata"],
        "annotations": {"entities": predicted_entities, "relations": []},
        "predictions": {"entities": predicted_entities, "relations": []}
    })

output_file = "mrbert_blind_submission.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=4)

print(f"🏆 BOOM! Final submission ready: {output_file}")

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading and combining Train + Dev sets for maximum learning...


TypeError: concat() got an unexpected keyword argument 'ignore_axis'

Baseline Evaluator

In [4]:
# =========================================================================================
# 📊 CELL 1: INDIVIDUAL MODEL BENCHMARKER
# Evaluates each base model's raw logits to see their individual strengths.
# =========================================================================================
import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import re
from collections import defaultdict
from typing import NamedTuple

print("Loading Individual Model Logits...")
LOGIT_DIR = "/kaggle/input/datasets/alexcristea72/3-logits/"
DEV_JSON = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"

p_mr  = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}mrbert_biomed_dev_logits.npy")), dim=-1).numpy()
p_deb = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}deberta_dev_logits.npy")), dim=-1).numpy()
p_rob = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}roberta_dev_logits.npy")), dim=-1).numpy()
p_bsc = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}bsc_bio_dev_logits.npy")), dim=-1).numpy()

models = {"MrBERT Biomed": p_mr, "DeBERTa-v3": p_deb, "Clinical RoBERTa": p_rob, "BSC-Bio-EHR": p_bsc}

# --- SCORING ENGINE ---
with open(DEV_JSON, "r", encoding="utf-8") as f: dev_data = json.load(f)
nlp = spacy.load("es_core_news_sm")

all_sentences = []
dev_eval_template = []
for doc in dev_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in nlp(raw_text).sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    for sr in sent_records: all_sentences.append(sr["text"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _enrich(entities, tp): return [{**e, "tokens": frozenset(i for i, (ts, te) in enumerate(tp) if ts >= e["start"] and te <= e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset

def evaluate_strict(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case["raw_text"])
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        g_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        p_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        used, matched, ug = set(), [], []
        for a in g_comps:
            j = next((j for j, b in enumerate(p_comps) if j not in used and a.type==b.type and a.start==b.start and a.end==b.end), None)
            if j is not None: matched.append((a, p_comps[j])); used.add(j)
            else: ug.append(a)
        up = [p_comps[j] for j in range(len(p_comps)) if j not in used]
        for g, _ in matched: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    
    f1s, metrics = [], {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        metrics[t] = f; f1s.append(f)
    metrics["Macro"] = sum(f1s) / 3
    return metrics

# --- RUN EVALUATION ---
inv_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}
print("\n" + "="*60)
for name, probs in models.items():
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3 # Apply MajorClaim boost
    t2p = {t: p for t, p in zip(all_sentences, preds)}
    
    cases = []
    for skel in dev_eval_template:
        ents = []
        for i, sr in enumerate(skel["sent_records"]):
            lbl = inv_map.get(t2p.get(sr["text"], 0), "None")
            if lbl != "None": ents.append({"id": f"T{i}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": lbl})
        cases.append({"raw_text": skel["raw_text"], "annotations": skel["annotations"], "predictions": {"entities": ents}})
    
    res = evaluate_strict(cases)
    print(f"🤖 {name.upper()}")
    print(f"   🥇 Macro F1   : {res['Macro']:.4f}")
    print(f"   📊 Premise    : {res['Premise']:.4f}")
    print(f"   📈 Claim      : {res['Claim']:.4f}")
    print(f"   👑 MajorClaim : {res['MajorClaim']:.4f}")
    print("-" * 60)

Loading Individual Model Logits...

🤖 MRBERT BIOMED
   🥇 Macro F1   : 0.7321
   📊 Premise    : 0.9005
   📈 Claim      : 0.8252
   👑 MajorClaim : 0.4706
------------------------------------------------------------
🤖 DEBERTA-V3
   🥇 Macro F1   : 0.5966
   📊 Premise    : 0.8640
   📈 Claim      : 0.7071
   👑 MajorClaim : 0.2187
------------------------------------------------------------
🤖 CLINICAL ROBERTA
   🥇 Macro F1   : 0.6724
   📊 Premise    : 0.8970
   📈 Claim      : 0.7867
   👑 MajorClaim : 0.3333
------------------------------------------------------------
🤖 BSC-BIO-EHR
   🥇 Macro F1   : 0.6303
   📊 Premise    : 0.8945
   📈 Claim      : 0.7963
   👑 MajorClaim : 0.2000
------------------------------------------------------------


The Meta-Learner (Logistic Regression)

In [10]:
# =========================================================================================
# 🧠 CELL 3 (REVISED): THE META-LEARNER (CROSS-VALIDATED STACKING)
# Uses ONLY the Dev Logits you already have to prevent Data Leakage!
# =========================================================================================
import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import re
from collections import defaultdict
from typing import NamedTuple
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

print("🧠 Initiating Cross-Validated Meta-Learner...")

# 1. LOAD YOUR SAVED DEV LOGITS (No extra training required!)
LOGIT_DIR = "/kaggle/input/datasets/alexcristea72/3-logits/"
try:
    p_mr  = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}mrbert_biomed_dev_logits.npy")), dim=-1).numpy()
    p_deb = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}deberta_dev_logits.npy")), dim=-1).numpy()
    p_rob = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}roberta_dev_logits.npy")), dim=-1).numpy()
    p_bsc = F.softmax(torch.tensor(np.load(f"{LOGIT_DIR}bsc_bio_dev_logits.npy")), dim=-1).numpy()
    print("✅ Successfully loaded all 4 Dev Logit files.")
except FileNotFoundError as e:
    print(f"❌ ERROR: Missing logit files. {e}")
    raise

# Stack features side-by-side (705 sentences, 16 features each)
X_dev_meta = np.hstack([p_mr, p_deb, p_rob, p_bsc])

# 2. EXTRACT TRUE LABELS FROM DEV JSON
DEV_JSON = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
with open(DEV_JSON, "r", encoding="utf-8") as f: dev_data = json.load(f)

nlp = spacy.load("es_core_news_sm")
y_dev = []
dev_texts = []
dev_eval_template = []

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}

for doc in dev_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    sent_records = [{"text": a["text"].strip(), "label": label_map[a["type"]], "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in nlp(raw_text).sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "label": 0, "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    for sr in sent_records:
        y_dev.append(sr["label"])
        dev_texts.append(sr["text"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

y_dev = np.array(y_dev)

# 3. TRAIN AND EVALUATE USING K-FOLD CROSS VALIDATION
print("🔄 Running 5-Fold Cross Validation to prevent overfitting...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Array to hold the "Out-of-Fold" predictions
oof_probs = np.zeros((len(y_dev), 4))

for train_idx, test_idx in skf.split(X_dev_meta, y_dev):
    X_train_fold, X_test_fold = X_dev_meta[train_idx], X_dev_meta[test_idx]
    y_train_fold = y_dev[train_idx]
    
    fold_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=2000, class_weight='balanced', random_state=42)
    fold_model.fit(X_train_fold, y_train_fold)
    
    oof_probs[test_idx] = fold_model.predict_proba(X_test_fold)

# Calculate final predictions from the out-of-fold probabilities
meta_preds = np.argmax(oof_probs, axis=-1)
meta_preds[oof_probs[:, 3] > 0.50] = 3 # MajorClaim safety net

# 4. OFFICIAL STRICT SCORING REBUILDER
print("⚖️ Calculating Official Strict F1 Score...")
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _enrich(entities, tp): return [{**e, "tokens": frozenset(i for i, (ts, te) in enumerate(tp) if ts >= e["start"] and te <= e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset

def evaluate_strict(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case["raw_text"])
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        g_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        p_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        used, matched, ug = set(), [], []
        for a in g_comps:
            j = next((j for j, b in enumerate(p_comps) if j not in used and a.type==b.type and a.start==b.start and a.end==b.end), None)
            if j is not None: matched.append((a, p_comps[j])); used.add(j)
            else: ug.append(a)
        up = [p_comps[j] for j in range(len(p_comps)) if j not in used]
        for g, _ in matched: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    
    f1s, metrics = [], {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        metrics[t] = f; f1s.append(f)
    metrics["Macro"] = sum(f1s) / 3
    return metrics

inv_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}
text_to_pred = {text: pred for text, pred in zip(dev_texts, meta_preds)}
cases = []
for skel in dev_eval_template:
    ents = []
    for i, sr in enumerate(skel["sent_records"]):
        lbl = inv_map.get(text_to_pred.get(sr["text"], 0), "None")
        if lbl != "None": ents.append({"id": f"T{i}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": lbl})
    cases.append({"raw_text": skel["raw_text"], "annotations": skel["annotations"], "predictions": {"entities": ents}})

res = evaluate_strict(cases)
print("\n" + "="*60)
print(f"🏆 META-LEARNER OFFICIAL CROSS-VALIDATED SCORE")
print(f"   🥇 Macro F1   : {res['Macro']:.4f}")
print(f"   📊 Premise    : {res['Premise']:.4f}")
print(f"   📈 Claim      : {res['Claim']:.4f}")
print(f"   👑 MajorClaim : {res['MajorClaim']:.4f}")
print("="*60)

# 5. BONUS: SHOW THE AI'S CHOSEN WEIGHTS
print("\n🔍 WHAT DID THE AI LEARN?")
final_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=2000, class_weight='balanced')
final_model.fit(X_dev_meta, y_dev)
print("When predicting MajorClaims, here is how much the AI trusted each model:")
print(f" - MrBERT Weight:  {final_model.coef_[3][3]:.2f}")
print(f" - DeBERTa Weight: {final_model.coef_[3][7]:.2f}")
print(f" - RoBERTa Weight: {final_model.coef_[3][11]:.2f}")
print(f" - BSC-Bio Weight: {final_model.coef_[3][15]:.2f}")

🧠 Initiating Cross-Validated Meta-Learner...
✅ Successfully loaded all 4 Dev Logit files.
🔄 Running 5-Fold Cross Validation to prevent overfitting...
⚖️ Calculating Official Strict F1 Score...

🏆 META-LEARNER OFFICIAL CROSS-VALIDATED SCORE
   🥇 Macro F1   : 0.6383
   📊 Premise    : 0.9083
   📈 Claim      : 0.8365
   👑 MajorClaim : 0.1702

🔍 WHAT DID THE AI LEARN?
When predicting MajorClaims, here is how much the AI trusted each model:
 - MrBERT Weight:  1.92
 - DeBERTa Weight: 1.98
 - RoBERTa Weight: 0.77
 - BSC-Bio Weight: -0.54


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

In [2]:
# =========================================================================================
# 🛠️ THE GRANDMASTER HEURISTICS (Post-Processing Evaluator)
# Tests if Merging Spans and Length Filtering improves your MrBERT F1 Score
# =========================================================================================
import subprocess, sys

# 1. Download Spanish grammar engine silently (Fixes the OSError)
print("Downloading SpaCy Spanish Engine...")
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import re
from collections import defaultdict
from typing import NamedTuple

print("Loading MrBERT Dev Logits...")
# Make sure the path matches where your numpy file actually is right now
# (If it's in the input folder, use: "/kaggle/input/datasets/alexcristea72/3-logits/mrbert_biomed_dev_logits.npy")
try:
    p_mr = F.softmax(torch.tensor(np.load("/kaggle/input/datasets/alexcristea72/3-logits/mrbert_biomed_dev_logits.npy")), dim=-1).numpy()
except FileNotFoundError:
    p_mr = F.softmax(torch.tensor(np.load("mrbert_biomed_dev_logits.npy")), dim=-1).numpy()

preds = np.argmax(p_mr, axis=-1)
preds[p_mr[:, 3] > 0.50] = 3 # The standard MajorClaim boost

DEV_JSON = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
with open(DEV_JSON, "r", encoding="utf-8") as f: dev_data = json.load(f)

# Now this will load perfectly!
nlp = spacy.load("es_core_news_sm")

all_sentences = []
dev_eval_template = []
for doc in dev_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in nlp(raw_text).sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    for sr in sent_records: all_sentences.append(sr["text"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

t2p = {t: p for t, p in zip(all_sentences, preds)}
inv_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}

# --- THE HEURISTICS ENGINE ---
cases = []
for skel in dev_eval_template:
    ents = []
    current_span = None
    
    for i, sr in enumerate(skel["sent_records"]):
        lbl = inv_map.get(t2p.get(sr["text"], 0), "None")
        
        # 1. THE LENGTH FILTER (Drop noise under 15 chars)
        if lbl != "None" and (sr["end"] - sr["start"]) < 15:
            lbl = "None" 

        if lbl == "None":
            if current_span:
                ents.append(current_span)
                current_span = None
        else:
            if not current_span:
                # Start a new span
                current_span = {"id": f"T{len(ents)}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": lbl}
            else:
                # 2. MERGE ADJACENT SPANS
                if current_span["type"] == lbl:
                    # Same label? Merge them!
                    current_span["end"] = sr["end"]
                    current_span["text"] = skel["raw_text"][current_span["start"]:current_span["end"]]
                else:
                    # Different label? Save old, start new.
                    ents.append(current_span)
                    current_span = {"id": f"T{len(ents)}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": lbl}
    
    if current_span:
        ents.append(current_span)
        
    cases.append({"raw_text": skel["raw_text"], "annotations": skel["annotations"], "predictions": {"entities": ents}})

# --- EVALUATION ---
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _enrich(entities, tp): return [{**e, "tokens": frozenset(i for i, (ts, te) in enumerate(tp) if ts >= e["start"] and te <= e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset

def evaluate_strict(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case["raw_text"])
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        g_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        p_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        used, matched, ug = set(), [], []
        for a in g_comps:
            j = next((j for j, b in enumerate(p_comps) if j not in used and a.type==b.type and a.start==b.start and a.end==b.end), None)
            if j is not None: matched.append((a, p_comps[j])); used.add(j)
            else: ug.append(a)
        up = [p_comps[j] for j in range(len(p_comps)) if j not in used]
        for g, _ in matched: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    
    f1s, metrics = [], {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        metrics[t] = f; f1s.append(f)
    metrics["Macro"] = sum(f1s) / 3
    return metrics

res = evaluate_strict(cases)
print("\n" + "="*60)
print(f"🤖 MRBERT (WITH HEURISTICS)")
print(f"   🥇 Macro F1   : {res['Macro']:.4f} (Original: 0.7321)")
print(f"   📊 Premise    : {res['Premise']:.4f}")
print(f"   📈 Claim      : {res['Claim']:.4f}")
print(f"   👑 MajorClaim : {res['MajorClaim']:.4f}")
print("=" * 60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 88.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading MrBERT Dev Logits...

🤖 MRBERT (WITH HEURISTICS)
   🥇 Macro F1   : 0.2594 (Original: 0.7321)
   📊 Premise    : 0.0500
   📈 Claim      : 0.2577
   👑 MajorClaim : 0.4706


In [3]:
# =========================================================================================
# 🛠️ POST-PROCESSING: LENGTH FILTER ONLY
# =========================================================================================
import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy
import re
from collections import defaultdict
from typing import NamedTuple

print("Loading MrBERT Dev Logits for Length Filter Test...")
try:
    p_mr = F.softmax(torch.tensor(np.load("/kaggle/input/datasets/alexcristea72/3-logits/mrbert_biomed_dev_logits.npy")), dim=-1).numpy()
except FileNotFoundError:
    p_mr = F.softmax(torch.tensor(np.load("mrbert_biomed_dev_logits.npy")), dim=-1).numpy()

preds = np.argmax(p_mr, axis=-1)
preds[p_mr[:, 3] > 0.50] = 3 

DEV_JSON = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
with open(DEV_JSON, "r", encoding="utf-8") as f: dev_data = json.load(f)

nlp = spacy.load("es_core_news_sm")
all_sentences, dev_eval_template = [], []

for doc in dev_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in nlp(raw_text).sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    for sr in sent_records: all_sentences.append(sr["text"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

t2p = {t: p for t, p in zip(all_sentences, preds)}
inv_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}

cases = []
for skel in dev_eval_template:
    ents = []
    e_id = 1
    for sr in skel["sent_records"]:
        lbl = inv_map.get(t2p.get(sr["text"], 0), "None")
        
        # --- THE LENGTH FILTER ---
        # If the model predicts an entity, but it's shorter than 10 characters, turn it to "None"
        if lbl != "None" and (sr["end"] - sr["start"]) < 10:
            lbl = "None"
            
        if lbl != "None":
            ents.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": lbl})
            e_id += 1
            
    cases.append({"raw_text": skel["raw_text"], "annotations": skel["annotations"], "predictions": {"entities": ents}})

# Evaluation setup
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _enrich(entities, tp): return [{**e, "tokens": frozenset(i for i, (ts, te) in enumerate(tp) if ts >= e["start"] and te <= e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset

def evaluate_strict(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case["raw_text"])
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        g_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        p_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        used, matched, ug = set(), [], []
        for a in g_comps:
            j = next((j for j, b in enumerate(p_comps) if j not in used and a.type==b.type and a.start==b.start and a.end==b.end), None)
            if j is not None: matched.append((a, p_comps[j])); used.add(j)
            else: ug.append(a)
        up = [p_comps[j] for j in range(len(p_comps)) if j not in used]
        for g, _ in matched: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    
    f1s, metrics = [], {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        metrics[t] = f; f1s.append(f)
    metrics["Macro"] = sum(f1s) / 3
    return metrics

res = evaluate_strict(cases)
print("\n" + "="*60)
print(f"🤖 MRBERT (LENGTH FILTER ONLY - < 10 Chars)")
print(f"   🥇 Macro F1   : {res['Macro']:.4f} (Original: 0.7321)")
print("=" * 60)

Loading MrBERT Dev Logits for Length Filter Test...

🤖 MRBERT (LENGTH FILTER ONLY - < 10 Chars)
   🥇 Macro F1   : 0.7321 (Original: 0.7321)


In [5]:
# =========================================================================================
# 📍 GRACE Track 1 — MrBERT Biomed Expert (POSITION FEATURES ADDED)
# Tests if adding [S_x_of_y] improves structural understanding of MajorClaims
# =========================================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys

# 1. Download Spanish grammar engine silently
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset

# =========================================================================================
# 0. LOCK THE UNIVERSE (Seed Everything)
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# =========================================================================================
# PART 1: THE DATA PARSER (Position Injection + Bi-Directional Context)
# =========================================================================================
def parse_grace_json_with_position(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        
        sent_records = [{"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": sent.start_char})
        
        sent_records.sort(key=lambda x: x["start"])
        total_sents = len(sent_records)
        
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            
            # --- THE MAGIC TRICK: POSITION INJECTION ---
            position_tag = f"[S_{i+1}_of_{total_sents}]"
            modified_text = f"{position_tag} {sr['text']}"
            
            records.append({
                "text": modified_text,                       # For Model Training
                "original_text": sr["text"],                 # For Strict Evaluation mapping
                "context": f"{prev_text} [SEP] {next_text}".strip(), 
                "label": sr["label"]
            })
            
    return pd.DataFrame(records)


# =========================================================================================
# PART 2: LOAD & ENCODE
# =========================================================================================
train_df = parse_grace_json_with_position("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json")
dev_df   = parse_grace_json_with_position("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json")

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

train_df["label"] = train_df["label"].map(label_map)
dev_df["label"] = dev_df["label"].map(label_map)

MODEL_NAME = "BSC-LT/MrBERT-biomed"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nDownloading MrBERT Biomed weights...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True, attn_implementation="eager"
)

def tokenize_fn(examples):
    # Updated to use "context" and increased max_length to fit the new tags and bidirectional context
    return tokenizer(examples["context"], examples["text"], padding="max_length", truncation=True, max_length=256, return_token_type_ids=False)

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)


# =========================================================================================
# PART 3: IN-MEMORY SCORING CACHE (Optimization)
# =========================================================================================
print("Pre-building Dev set evaluation template...")
nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json", "r", encoding="utf-8"))

dev_eval_template = []
for doc in dev_raw_data:
    raw_text = doc["raw_text"]
    annotations = doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    spacy_doc = nlp_cache(raw_text)

    sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
    for sent in spacy_doc.sents:
        s, e = sent.start_char, sent.end_char
        if not any(max(s, a) < min(e, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": s, "end": e})

    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"id": doc["id"], "raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})


# =========================================================================================
# PART 4: THE OFFICIAL SCORING ENGINE (Stripped down to Strict rules)
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text: str): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _span_token_set(token_positions, start, end): return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)
def _enrich(entities, token_positions): return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _exact_span(a, b): return a.type == b.type and a.start == b.start and a.end == b.end

def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    unmatched_b = [items_b[j] for j in range(len(items_b)) if j not in used]
    return matched, unmatched_a, unmatched_b

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * p * r / (p + r) if (p + r) else 0.0

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        matched, unmatched_gold, unmatched_pred = _greedy_match(gold_comps, pred_comps, _exact_span)
        for g, _ in matched: tp_s[g.type] += 1
        for r in unmatched_pred: fp_s[r.type] += 1
        for r in unmatched_gold: fn_s[r.type] += 1
    f1s, metrics = [], {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        f = _prf(tp_s[t], fp_s[t], fn_s[t])
        metrics[t] = f
        f1s.append(f)
    metrics["Macro_F1"] = sum(f1s) / len(f1s) if f1s else 0.0
    return metrics


# =========================================================================================
# PART 5: THE LIVE EPOCH CALLBACK
# =========================================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3

    # BUG FIX: Score using the original text, NOT the modified position text!
    text_to_pred = {orig_text: pred for orig_text, pred in zip(dev_df["original_text"], preds)}
    cases = []

    for template in dev_eval_template:
        predicted_entities, e_id = [], 1
        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
                e_id += 1
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})

    metrics = evaluate_strict_f1(cases)
    print("\n--- EPOCH EVALUATION ---")
    print(f"🥇 Macro F1   : {metrics['Macro_F1']:.4f}")
    print(f"📊 Premise    : {metrics['Premise']:.4f}")
    print(f"📈 Claim      : {metrics['Claim']:.4f}")
    print(f"👑 MajorClaim : {metrics['MajorClaim']:.4f}")
    print("------------------------\n")
    return {"strict_macro_f1": metrics["Macro_F1"]}


# =========================================================================================
# PART 6: TRAINING SETUP & DIFFERENTIAL OPTIMIZER
# =========================================================================================
counts = np.bincount(train_df["label"].values, minlength=4).astype(float)
weights = ((counts.sum() / (len(counts) * counts)) / (counts.sum() / (len(counts) * counts)).mean()).tolist()
weights[3] = 10.0 # MajorClaim boost

head_params = [p for n, p in model.named_parameters() if "classifier" in n]
base_params = [p for n, p in model.named_parameters() if "classifier" not in n]
custom_optimizer = torch.optim.AdamW([{"params": base_params, "lr": 1e-5}, {"params": head_params, "lr": 1e-4}], weight_decay=0.05)

training_args = TrainingArguments(
    output_dir="./mrbert_position_grace", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=15, fp16=True,
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=42, data_seed=42
)

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

trainer = WeightedCETrainer(
    model=model, args=training_args, train_dataset=train_dataset, eval_dataset=dev_dataset,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    optimizers=(custom_optimizer, None)
)

print("\n🚀 Starting Position-Aware Training Loop...")
trainer.train()

# =========================================================================================
# PART 7: EXTRACTION
# =========================================================================================
print("\nExtracting Golden Predictions...")
logits = trainer.predict(dev_dataset).predictions
np.save("mrbert_position_dev_logits.npy", logits)
print("✅ Saved mrbert_position_dev_logits.npy!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 84.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/37.0M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- EPOCH EVALUATION ---
🥇 Macro F1   : 0.6407
📊 Premise    : 0.8957
📈 Claim      : 0.8159
👑 MajorClaim : 0.2105
------------------------



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Extracting Golden Predictions...



--- EPOCH EVALUATION ---
🥇 Macro F1   : 0.6691
📊 Premise    : 0.8636
📈 Claim      : 0.8103
👑 MajorClaim : 0.3333
------------------------

✅ Saved mrbert_position_dev_logits.npy!


In [1]:
# =========================================================================================
# 🧬 GRACE Track 1 — Multi-Seed Multi-Epoch Logit Extraction Pipeline
# Captures and saves raw logits at EVERY epoch for ALL seeds to maximize ensemble choices.
# =========================================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import Dataset

# --- CONFIGURATION HUB ---
SEEDS = [42, 21, 100]
TOTAL_EPOCHS = 5
MODEL_NAME = "BSC-LT/MrBERT-biomed"
FILE_PREFIX = "mrbert_biomed"
# -------------------------

# =========================================================================================
# 0. GLOBAL SEED LOCK
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# =========================================================================================
# PART 1: THE DATA PARSER (Context Aware)
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    nlp = spacy.load("es_core_news_sm")
    records = []

    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc["annotations"].get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]

        spacy_doc = nlp(raw_text)
        all_sents = list(spacy_doc.sents)
        sent_records = []

        for ann in annotations:
            sent_records.append({
                "text": ann["text"].strip(),
                "label": ann["type"],
                "start": ann["start"]
            })

        for sent in all_sents:
            s, e = sent.start_char, sent.end_char
            if not any(max(s, a) < min(e, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({
                        "text": sent.text.strip(),
                        "label": "None",
                        "start": s
                    })

        sent_records.sort(key=lambda x: x["start"])

        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({
                "text": sr["text"],
                "prev_text": prev_text,
                "label": sr["label"]
            })

    return pd.DataFrame(records)

# =========================================================================================
# PART 2: INITIAL DATA LOAD & CACHE BUILD
# =========================================================================================
print("Parsing Datasets...")
train_df = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json")
dev_df   = parse_grace_json("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json")

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

train_df["label"] = train_df["label"].map(label_map)
dev_df["label"] = dev_df["label"].map(label_map)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["prev_text"], examples["text"],
        padding="max_length", truncation=True, max_length=192,
        return_token_type_ids=False
    )

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
dev_dataset   = Dataset.from_pandas(dev_df).map(tokenize_fn, batched=True)

print("Pre-building Evaluation Cache Template...")
nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open("/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json", "r", encoding="utf-8"))

dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
    spacy_doc = nlp_cache(raw_text)

    sent_records = [{"text": ann["text"].strip(), "start": ann["start"], "end": ann["end"]} for ann in annotations]
    for sent in spacy_doc.sents:
        s, e = sent.start_char, sent.end_char
        if not any(max(s, a) < min(e, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": s, "end": e})

    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"id": doc["id"], "raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

# =========================================================================================
# PART 3: EVALUATION & LOSS CORE
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text: str): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _span_token_set(token_positions, start, end): return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)
def _enrich(entities, token_positions): return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _exact_span(a, b): return a.type == b.type and a.start == b.start and a.end == b.end

def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    unmatched_b = [items_b[j] for j in range(len(items_b)) if j not in used]
    return matched, unmatched_a, unmatched_b

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * p * r / (p + r) if (p + r) else 0.0

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        matched, unmatched_gold, unmatched_pred = _greedy_match(gold_comps, pred_comps, _exact_span)
        for g, _ in matched: tp_s[g.type] += 1
        for r in unmatched_pred: fp_s[r.type] += 1
        for r in unmatched_gold: fn_s[r.type] += 1
    f1s = []
    metrics = {}
    for t in ["Premise", "Claim", "MajorClaim"]:
        f = _prf(tp_s[t], fp_s[t], fn_s[t])
        metrics[t] = f; f1s.append(f)
    metrics["Macro_F1"] = sum(f1s) / len(f1s) if f1s else 0.0
    return metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3

    text_to_pred = {text: pred for text, pred in zip(dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities, e_id = [], 1
        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
                e_id += 1
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    metrics = evaluate_strict_f1(cases)
    return {"strict_macro_f1": metrics["Macro_F1"]}

counts = np.bincount(train_df["label"].values, minlength=4).astype(float)
weights = ((counts.sum() / (len(counts) * counts)) / (counts.sum() / (len(counts) * counts)).mean()).tolist()
weights[3] = 10.0

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights, dtype=outputs.get("logits").dtype, device=outputs.get("logits").device)
        return (F.cross_entropy(outputs.get("logits"), labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.get("logits"), labels, weight=w)

# =========================================================================================
# PART 4: THE INTERCEPTOR CALLBACK (EPOCH LOGIT SAVER)
# =========================================================================================
class EpochLogitSaverCallback(TrainerCallback):
    def __init__(self, eval_dataset, seed, file_prefix):
        self.eval_dataset = eval_dataset
        self.seed = seed
        self.file_prefix = file_prefix
        self.trainer_ref = None

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        if self.trainer_ref is not None:
            print(f"\n💾 [Callback Triggered] Intercepting Epoch {epoch} for Seed {self.seed}...")
            # Predict out-of-fold probabilities right now
            raw_preds = self.trainer_ref.predict(self.eval_dataset)
            epoch_logits = raw_preds.predictions
            
            # Save array directly to workspace
            output_name = f"{self.file_prefix}_seed{self.seed}_epoch{epoch}_logits.npy"
            np.save(output_name, epoch_logits)
            print(f"✅ Successfully exported arrays -> {output_name}")

# =========================================================================================
# PART 5: MULTI-SEED OUTER EXECUTION LOOP
# =========================================================================================
for seed in SEEDS:
    print("\n" + "="*70)
    print(f"🚀 INITIALIZING RUN LOOP: SEED {seed}")
    print("="*70)
    
    seed_everything(seed)
    
    # Initialize a completely fresh copy of the model to clear downstream learning
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True, attn_implementation="eager"
    )
    
    head_params = [p for n, p in model.named_parameters() if "classifier" in n]
    base_params = [p for n, p in model.named_parameters() if "classifier" not in n]
    custom_optimizer = torch.optim.AdamW([{"params": base_params, "lr": 1e-5}, {"params": head_params, "lr": 1e-4}], weight_decay=0.05)

    training_args = TrainingArguments(
        output_dir=f"./working_dir_seed_{seed}",
        eval_strategy="epoch", save_strategy="epoch",
        warmup_steps=100, lr_scheduler_type="cosine",
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        num_train_epochs=TOTAL_EPOCHS, fp16=True,
        load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
        save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
        seed=seed, data_seed=seed
    )
    
    # Instantiate the custom callback for this specific run state
    logit_saver = EpochLogitSaverCallback(eval_dataset=dev_dataset, seed=seed, file_prefix=FILE_PREFIX)
    
    trainer = WeightedCETrainer(
        model=model, args=training_args, train_dataset=train_dataset, eval_dataset=dev_dataset,
        compute_metrics=compute_metrics, 
        callbacks=[logit_saver, EarlyStoppingCallback(early_stopping_patience=3)],
        optimizers=(custom_optimizer, None)
    )
    
    # Pass trainer reference directly into the callback object structure
    logit_saver.trainer_ref = trainer
    
    print(f"Starting Training Matrix loop for Seed {seed}...")
    trainer.train()
    
    # RAM Hygiene: Clean up variables and flush memory slots before starting the next seed
    del model, trainer, custom_optimizer
    torch.cuda.empty_cache()

print("\n🎉 ALL SEEDS AND ALL EPOCHS CAPTURED! Check your working directory for your .npy matrix files.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 78.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Parsing Datasets...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/37.0M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

...


💾 [Callback Triggered] Intercepting Epoch 1 for Seed 42...
✅ Successfully exported arrays -> mrbert_biomed_seed42_epoch1_logits.npy


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 [Callback Triggered] Intercepting Epoch 2 for Seed 42...
✅ Successfully exported arrays -> mrbert_biomed_seed42_epoch2_logits.npy


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
ATAPIHTOPAO{Bew[obearo;ibagro[bgoaboabgw]]}

In [ ]:
# =========================================================================================
# 🏆 THE TWO-STAGE AUTO-RUNNER (Self-Tuning & Blind Logit Extraction)
# Phase 1: Finds best epoch on Dev. 
# Phase 2: Trains on Train+Dev for (Best_Epoch + 1) and saves blind logits per epoch.
# =========================================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
import gc
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import Dataset

# --- CONFIGURATION HUB ---
SEED = 42
MODEL_NAME = "BSC-LT/MrBERT-biomed" # Change to assigned model
FILE_PREFIX = "mrbert_biomed"       # E.g., mrbert_biomed, deberta, bsc_bio
EXTRA_EPOCHS_FOR_FULL_FIT = 1       # Adds +1 to the best found epoch

TRAIN_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json"
DEV_PATH   = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
TRAIN_DEV_PATH = "merged_train_dev_full_fit.json"
BLIND_TEST_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/GRACE_2026_blind_test/track_1_blind_test.json"
# -------------------------

# =========================================================================================
# 0. SEED LOCK
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# =========================================================================================
# 1. PARSERS
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": sent.start_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": sr["label"]})
    return pd.DataFrame(records)

def parse_blind_test(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        spacy_doc = nlp(raw_text)
        sent_records = []
        for sent in spacy_doc.sents:
            if len(sent.text.strip()) > 10:
                sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": 0})
    return pd.DataFrame(records)
# =========================================================================================
# 2. DATA PREPARATION FOR BOTH PHASES (BUG FIXED: In-Memory Merge)
# =========================================================================================
print("Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...")
p1_train_df = parse_grace_json(TRAIN_PATH)
p1_dev_df   = parse_grace_json(DEV_PATH)

# 🐛 THE FIX: Merge them dynamically in memory instead of loading the old JSON!
p2_train_df = pd.concat([p1_train_df, p1_dev_df], ignore_index=True)

p2_test_df  = parse_blind_test(BLIND_TEST_PATH)

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

p1_train_df["label"] = p1_train_df["label"].map(label_map)
p1_dev_df["label"]   = p1_dev_df["label"].map(label_map)
p2_train_df["label"] = p2_train_df["label"] # Labels already mapped during concat

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_fn(examples):
    return tokenizer(examples["prev_text"], examples["text"], padding="max_length", truncation=True, max_length=192, return_token_type_ids=False)

p1_train_ds = Dataset.from_pandas(p1_train_df).map(tokenize_fn, batched=True)
p1_dev_ds   = Dataset.from_pandas(p1_dev_df).map(tokenize_fn, batched=True)
p2_train_ds = Dataset.from_pandas(p2_train_df).map(tokenize_fn, batched=True)
p2_test_ds  = Dataset.from_pandas(p2_test_df).map(tokenize_fn, batched=True)

nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open(DEV_PATH, "r", encoding="utf-8"))
dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in spacy_doc.sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"id": doc["id"], "raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

# =========================================================================================
# 3. EVALUATION ENGINE (For Phase 1)
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text: str): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _span_token_set(token_positions, start, end): return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)
def _enrich(entities, token_positions): return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _exact_span(a, b): return a.type == b.type and a.start == b.start and a.end == b.end

def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    unmatched_b = [items_b[j] for j in range(len(items_b)) if j not in used]
    return matched, unmatched_a, unmatched_b

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        matched, unmatched_gold, unmatched_pred = _greedy_match(gold_comps, pred_comps, _exact_span)
        for g, _ in matched: tp_s[g.type] += 1
        for r in unmatched_pred: fp_s[r.type] += 1
        for r in unmatched_gold: fn_s[r.type] += 1
    f1s = []
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) else 0.0)
    return {"Macro_F1": sum(f1s) / len(f1s) if f1s else 0.0}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3
    text_to_pred = {text: pred for text, pred in zip(p1_dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities, e_id = [], 1
        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
                e_id += 1
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    return {"strict_macro_f1": evaluate_strict_f1(cases)["Macro_F1"]}

# Phase 1 Weight Calculator
counts_p1 = np.bincount(p1_train_df["label"].values, minlength=4).astype(float)
weights_p1 = ((counts_p1.sum() / (len(counts_p1) * counts_p1)) / (counts_p1.sum() / (len(counts_p1) * counts_p1)).mean()).tolist()
weights_p1[3] = 10.0

class WeightedCETrainer_P1(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p1, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# Phase 1 Best Epoch Tracker Callback
class BestEpochTracker(TrainerCallback):
    def __init__(self):
        self.best_f1 = 0.0
        self.best_epoch = 1

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_f1 = metrics.get("eval_strict_macro_f1", 0.0)
        if current_f1 > self.best_f1:
            self.best_f1 = current_f1
            self.best_epoch = round(state.epoch)

# =========================================================================================
# 🚀 PHASE 1: SEARCH FOR BEST EPOCH
# =========================================================================================
print("\n" + "="*80)
print(f"🔍 PHASE 1: SEARCHING FOR OPTIMAL EPOCH (SEED {SEED})")
print("="*80)

model_p1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" in n]
base_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" not in n]
optim_p1 = torch.optim.AdamW([{"params": base_params_p1, "lr": 1e-5}, {"params": head_params_p1, "lr": 1e-4}], weight_decay=0.05)

args_p1 = TrainingArguments(
    output_dir="./p1_search", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=15, fp16=True,
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=SEED, data_seed=SEED
)

tracker = BestEpochTracker()
trainer_p1 = WeightedCETrainer_P1(
    model=model_p1, args=args_p1, train_dataset=p1_train_ds, eval_dataset=p1_dev_ds,
    compute_metrics=compute_metrics, callbacks=[tracker, EarlyStoppingCallback(early_stopping_patience=4)], optimizers=(optim_p1, None)
)

trainer_p1.train()

FOUND_BEST_EPOCH = tracker.best_epoch
TARGET_P2_EPOCHS = FOUND_BEST_EPOCH + EXTRA_EPOCHS_FOR_FULL_FIT

print(f"\n🎯 PHASE 1 COMPLETE! Best Epoch Found: {FOUND_BEST_EPOCH} (Score: {tracker.best_f1:.4f})")
print(f"➕ Adding {EXTRA_EPOCHS_FOR_FULL_FIT} for Full-Fit. Phase 2 Target: {TARGET_P2_EPOCHS} Epochs.")

# --- MEMORY WIPE ---
del model_p1, trainer_p1, optim_p1
gc.collect()
torch.cuda.empty_cache()

# =========================================================================================
# 🚀 PHASE 2: PRODUCTION FULL-FIT & BLIND LOGIT EXTRACTION
# =========================================================================================
print("\n" + "="*80)
print(f"🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED {SEED} | {TARGET_P2_EPOCHS} EPOCHS)")
print("="*80)

# Phase 2 Weights
counts_p2 = np.bincount(p2_train_df["label"].values, minlength=4).astype(float)
weights_p2 = ((counts_p2.sum() / (len(counts_p2) * counts_p2)) / (counts_p2.sum() / (len(counts_p2) * counts_p2)).mean()).tolist()
weights_p2[3] = 10.0

class WeightedCETrainer_P2(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p2, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# Phase 2 Blind Logit Saver Callback
class BlindEpochLogitSaverCallback(TrainerCallback):
    def __init__(self, test_dataset, seed, file_prefix):
        self.test_dataset = test_dataset
        self.seed = seed
        self.file_prefix = file_prefix
        self.trainer_ref = None

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        if self.trainer_ref is not None:
            print(f"\n🔮 Extracting Blind Predictions for Phase 2, Epoch {epoch}...")
            raw_preds = self.trainer_ref.predict(self.test_dataset)
            blind_logits = raw_preds.predictions
            output_name = f"{self.file_prefix}_seed{self.seed}_epoch{epoch}_blind_logits.npy"
            np.save(output_name, blind_logits)
            print(f"💾 Saved: {output_name}")

# Init Fresh Model for Full-Fit
model_p2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" in n]
base_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" not in n]
optim_p2 = torch.optim.AdamW([{"params": base_params_p2, "lr": 1e-5}, {"params": head_params_p2, "lr": 1e-4}], weight_decay=0.05)

args_p2 = TrainingArguments(
    output_dir="./p2_production", eval_strategy="no", save_strategy="no",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16, 
    per_device_eval_batch_size=32, num_train_epochs=TARGET_P2_EPOCHS, fp16=True, 
    report_to="none", label_smoothing_factor=0.15, seed=SEED, data_seed=SEED
)

blind_saver = BlindEpochLogitSaverCallback(test_dataset=p2_test_ds, seed=SEED, file_prefix=FILE_PREFIX)

trainer_p2 = WeightedCETrainer_P2(
    model=model_p2, args=args_p2, train_dataset=p2_train_ds, 
    callbacks=[blind_saver], optimizers=(optim_p2, None)
)
blind_saver.trainer_ref = trainer_p2

trainer_p2.train()

print(f"\n🎉 TWO-STAGE RUN COMPLETE FOR SEED {SEED}! Check your output folder for the blind test .npy files.")

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...


Map:   0%|          | 0/4584 [00:00<?, ? examples/s]

Map:   0%|          | 0/705 [00:00<?, ? examples/s]

Map:   0%|          | 0/5289 [00:00<?, ? examples/s]

Map:   0%|          | 0/26640 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Strict Macro F1
1,No log,0.523224,0.679882
2,0.822953,0.512002,0.672859
3,0.822953,0.414975,0.643753
4,0.405344,0.537224,0.687764


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# =========================================================================================
# 🏆 THE DEBERTA TWO-STAGE AUTO-RUNNER (Self-Tuning & Blind Logit Extraction)
# Strategy: Bi-Directional Context [SEP] Window, LayerNorm Hacks, 32-bit Stability
# Phase 1: Finds best epoch on Dev using Train only.
# Phase 2: Trains on Train+Dev for (Best_Epoch + 1) and saves blind logits per epoch.
# =========================================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "sentencepiece"], check=False)
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
import gc
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import Dataset
from huggingface_hub import hf_hub_download

# --- CONFIGURATION HUB ---
SEED = 42
MODEL_NAME = "microsoft/mdeberta-v3-base" 
FILE_PREFIX = "deberta_v3"       
EXTRA_EPOCHS_FOR_FULL_FIT = 1       

TRAIN_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json"
DEV_PATH   = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
BLIND_TEST_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/GRACE_2026_blind_test/track_1_blind_test.json"
# -------------------------

# =========================================================================================
# 0. SEED LOCK
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# =========================================================================================
# 1. PARSERS (Bi-Directional Context Edition)
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": sent.start_char})
        sent_records.sort(key=lambda x: x["start"])
        
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            context_string = f"{prev_text} [SEP] {next_text}".strip()
            records.append({"text": sr["text"], "context": context_string, "label": sr["label"]})
    return pd.DataFrame(records)

def parse_blind_test(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        spacy_doc = nlp(raw_text)
        sent_records = []
        for sent in spacy_doc.sents:
            if len(sent.text.strip()) > 10:
                sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
        sent_records.sort(key=lambda x: x["start"])
        
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            context_string = f"{prev_text} [SEP] {next_text}".strip()
            records.append({"text": sr["text"], "context": context_string, "label": 0})
    return pd.DataFrame(records)

# =========================================================================================
# 2. DATA PREPARATION FOR BOTH PHASES (BUG FIXED: Map Labels before Merging)
# =========================================================================================
print("Parsing and tokenizing datasets structural trees...")
p1_train_df = parse_grace_json(TRAIN_PATH)
p1_dev_df   = parse_grace_json(DEV_PATH)
p2_test_df  = parse_blind_test(BLIND_TEST_PATH)

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

# Map text objects to array integers FIRST to avoid downstream Cast exceptions
p1_train_df["label"] = p1_train_df["label"].map(label_map)
p1_dev_df["label"]   = p1_dev_df["label"].map(label_map)

# Concat safely after transformation mapping
p2_train_df = pd.concat([p1_train_df, p1_dev_df], ignore_index=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_fn(examples):
    return tokenizer(examples["context"], examples["text"], padding="max_length", truncation=True, max_length=256, return_token_type_ids=False)

p1_train_ds = Dataset.from_pandas(p1_train_df).map(tokenize_fn, batched=True)
p1_dev_ds   = Dataset.from_pandas(p1_dev_df).map(tokenize_fn, batched=True)
p2_train_ds = Dataset.from_pandas(p2_train_df).map(tokenize_fn, batched=True)
p2_test_ds  = Dataset.from_pandas(p2_test_df).map(tokenize_fn, batched=True)

nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open(DEV_PATH, "r", encoding="utf-8"))
dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in spacy_doc.sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

# =========================================================================================
# 3. EVALUATION ENGINE (Phase 1)
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text: str): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _span_token_set(token_positions, start, end): return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)
def _enrich(entities, token_positions): return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset

def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    return matched, unmatched_a, [items_b[j] for j in range(len(items_b)) if j not in used]

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tokens = _tokenize(case["raw_text"])
        gold_ents = _enrich(case["annotations"]["entities"], tokens)
        pred_ents = _enrich(case["predictions"]["entities"], tokens)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        m, ug, up = _greedy_match(gold_comps, pred_comps, lambda a, b: a.type == b.type and a.start == b.start and a.end == b.end)
        for g, _ in m: tp_s[g.type] += 1
        for r in up: fp_s[r.type] += 1
        for r in ug: fn_s[r.type] += 1
    f1s = []
    for t in ["Premise", "Claim", "MajorClaim"]:
        tp, fp, fn = tp_s[t], fp_s[t], fn_s[t]
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) else 0.0)
    return {"Macro_F1": sum(f1s) / 3}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3
    text_to_pred = {text: pred for text, pred in zip(p1_dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities = []
        for i, sr in enumerate(template["sent_records"]):
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{i}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    res = evaluate_strict_f1(cases)
    print(f"\n--- DEBERTA VALIDATION SEARCH METRICS ---\n🥇 Strict Macro F1: {res['Macro_F1']:.4f}\n")
    return {"strict_macro_f1": res["Macro_F1"]}

counts_p1 = np.bincount(p1_train_df["label"].values, minlength=4).astype(float)
weights_p1 = ((counts_p1.sum() / (len(counts_p1) * counts_p1)) / (counts_p1.sum() / (len(counts_p1) * counts_p1)).mean()).tolist()
weights_p1[3] = 10.0

class WeightedCETrainer_P1(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p1, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

class BestEpochTracker(TrainerCallback):
    def __init__(self):
        self.best_f1 = 0.0
        self.best_epoch = 1
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_f1 = metrics.get("eval_strict_macro_f1", 0.0)
        if current_f1 > self.best_f1:
            self.best_f1 = current_f1
            self.best_epoch = round(state.epoch)

# =========================================================================================
# 🚀 PHASE 1: SEARCH FOR BEST EPOCH
# =========================================================================================
print("\n" + "="*80)
print(f"🔍 PHASE 1: SEARCHING FOR OPTIMAL CONVERGENCE EPOCH (SEED {SEED})")
print("="*80)

config_p1 = AutoConfig.from_pretrained(MODEL_NAME, num_labels=4)
model_p1  = AutoModelForSequenceClassification.from_config(config_p1)
ckpt_path_p1 = hf_hub_download(repo_id=MODEL_NAME, filename="pytorch_model.bin")
raw_sd_p1    = torch.load(ckpt_path_p1, map_location="cpu")
remapped_sd_p1 = {k.replace("LayerNorm.gamma", "LayerNorm.weight").replace("LayerNorm.beta", "LayerNorm.bias"): v for k, v in raw_sd_p1.items()}
model_p1.load_state_dict(remapped_sd_p1, strict=False)

head_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" in n]
base_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" not in n]
optim_p1 = torch.optim.AdamW([{"params": base_params_p1, "lr": 1e-5}, {"params": head_params_p1, "lr": 1e-4}], weight_decay=0.05)

args_p1 = TrainingArguments(
    output_dir="./p1_search", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", 
    per_device_train_batch_size=8, gradient_accumulation_steps=2, # Safe VRAM profile
    per_device_eval_batch_size=32, num_train_epochs=15, 
    fp16=False, # Mandatory 32-bit DeBERTa protection profile
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=SEED, data_seed=SEED
)

tracker = BestEpochTracker()
trainer_p1 = WeightedCETrainer_P1(
    model=model_p1, args=args_p1, train_dataset=p1_train_ds, eval_dataset=p1_dev_ds,
    compute_metrics=compute_metrics, callbacks=[tracker, EarlyStoppingCallback(early_stopping_patience=4)], optimizers=(optim_p1, None)
)

trainer_p1.train()

FOUND_BEST_EPOCH = tracker.best_epoch
TARGET_P2_EPOCHS = FOUND_BEST_EPOCH + EXTRA_EPOCHS_FOR_FULL_FIT

print(f"\n🎯 PHASE 1 COMPLETE! Optimal Convergence Point: Epoch {FOUND_BEST_EPOCH} (F1 Score: {tracker.best_f1:.4f})")
print(f"➕ Adding +{EXTRA_EPOCHS_FOR_FULL_FIT} for combined footprint scale. Phase 2 Target: {TARGET_P2_EPOCHS} Epochs.")

# Wipe variables and clean cache pathways before re-initialization
del model_p1, trainer_p1, optim_p1
gc.collect()
torch.cuda.empty_cache()

# =========================================================================================
# 🚀 PHASE 2: PRODUCTION FULL-FIT & BLIND LOGIT EXTRACTION
# =========================================================================================
print("\n" + "="*80)
print(f"🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED {SEED} | {TARGET_P2_EPOCHS} TOTAL EPOCHS)")
print("="*80)

counts_p2 = np.bincount(p2_train_df["label"].values, minlength=4).astype(float)
weights_p2 = ((counts_p2.sum() / (len(counts_p2) * counts_p2)) / (counts_p2.sum() / (len(counts_p2) * counts_p2)).mean()).tolist()
weights_p2[3] = 10.0

class WeightedCETrainer_P2(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p2, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

class BlindEpochLogitSaverCallback(TrainerCallback):
    def __init__(self, test_dataset, seed, file_prefix):
        self.test_dataset = test_dataset
        self.seed = seed
        self.file_prefix = file_prefix
        self.trainer_ref = None

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        if self.trainer_ref is not None:
            print(f"\n🔮 [Callback Executing] Processing Blind Test Matrices for Epoch {epoch}...")
            raw_preds = self.trainer_ref.predict(self.test_dataset)
            blind_logits = raw_preds.predictions
            output_name = f"{self.file_prefix}_seed{self.seed}_epoch{epoch}_blind_logits.npy"
            np.save(output_name, blind_logits)
            print(f"💾 Saved raw float structures directly to output -> {output_name}")

config_p2 = AutoConfig.from_pretrained(MODEL_NAME, num_labels=4)
model_p2  = AutoModelForSequenceClassification.from_config(config_p2)
ckpt_path_p2 = hf_hub_download(repo_id=MODEL_NAME, filename="pytorch_model.bin")
raw_sd_p2    = torch.load(ckpt_path_p2, map_location="cpu")
remapped_sd_p2 = {k.replace("LayerNorm.gamma", "LayerNorm.weight").replace("LayerNorm.beta", "LayerNorm.bias"): v for k, v in raw_sd_p2.items()}
model_p2.load_state_dict(remapped_sd_p2, strict=False)

head_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" in n]
base_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" not in n]
optim_p2 = torch.optim.AdamW([{"params": base_params_p2, "lr": 1e-5}, {"params": head_params_p2, "lr": 1e-4}], weight_decay=0.05)

args_p2 = TrainingArguments(
    output_dir="./p2_production", eval_strategy="no", save_strategy="no",
    warmup_steps=100, lr_scheduler_type="cosine", 
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    per_device_eval_batch_size=32, num_train_epochs=TARGET_P2_EPOCHS, fp16=False, 
    report_to="none", label_smoothing_factor=0.15, seed=SEED, data_seed=SEED
)

blind_saver = BlindEpochLogitSaverCallback(test_dataset=p2_test_ds, seed=SEED, file_prefix=FILE_PREFIX)

trainer_p2 = WeightedCETrainer_P2(
    model=model_p2, args=args_p2, train_dataset=p2_train_ds, 
    callbacks=[blind_saver], optimizers=(optim_p2, None)
)
blind_saver.trainer_ref = trainer_p2

trainer_p2.train()

print(f"\n🎉 EXHALE! TWO-STAGE AUTO-RUN FOR {FILE_PREFIX.upper()} COMPLETE!")
print("Collect your saved .npy arrays from your working output directory and send them for final ensembling fusion.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 81.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Parsing and tokenizing datasets structural trees...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Map:   0%|          | 0/4584 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- DEBERTA VALIDATION SEARCH METRICS ---
🥇 Strict Macro F1: 0.6254



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- DEBERTA VALIDATION SEARCH METRICS ---
🥇 Strict Macro F1: 0.6122



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
# =========================================================================================
# 🏆 THE TWO-STAGE AUTO-RUNNER (Self-Tuning & Blind Logit Extraction)
# Phase 1: Finds best epoch on Dev. 
# Phase 2: Trains on Train+Dev for (Best_Epoch + 1) and saves blind logits per epoch.
# =========================================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
import gc
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import Dataset

# --- CONFIGURATION HUB ---
SEED = 42
MODEL_NAME = "BSC-LT/MrBERT-biomed" # Change to assigned model
FILE_PREFIX = "mrbert_biomed"       # E.g., mrbert_biomed, deberta, bsc_bio
EXTRA_EPOCHS_FOR_FULL_FIT = 1       # Adds +1 to the best found epoch

TRAIN_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json"
DEV_PATH   = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
TRAIN_DEV_PATH = "merged_train_dev_full_fit.json"
BLIND_TEST_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/GRACE_2026_blind_test/track_1_blind_test.json"
# -------------------------

# =========================================================================================
# 0. SEED LOCK
# =========================================================================================
def seed_everything(seed=100):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# =========================================================================================
# 1. PARSERS
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": sent.start_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": sr["label"]})
    return pd.DataFrame(records)

def parse_blind_test(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        spacy_doc = nlp(raw_text)
        sent_records = []
        for sent in spacy_doc.sents:
            if len(sent.text.strip()) > 10:
                sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": 0})
    return pd.DataFrame(records)

# =========================================================================================
# 2. DATA PREPARATION FOR BOTH PHASES (BUG FIXED: In-Memory Merge)
# =========================================================================================
print("Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...")
p1_train_df = parse_grace_json(TRAIN_PATH)
p1_dev_df   = parse_grace_json(DEV_PATH)
p2_test_df  = parse_blind_test(BLIND_TEST_PATH)

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

# Ensure mapping is done BEFORE merge!
p1_train_df["label"] = p1_train_df["label"].map(label_map)
p1_dev_df["label"]   = p1_dev_df["label"].map(label_map)
p2_train_df = pd.concat([p1_train_df, p1_dev_df], ignore_index=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_fn(examples):
    return tokenizer(examples["prev_text"], examples["text"], padding="max_length", truncation=True, max_length=192, return_token_type_ids=False)

p1_train_ds = Dataset.from_pandas(p1_train_df).map(tokenize_fn, batched=True)
p1_dev_ds   = Dataset.from_pandas(p1_dev_df).map(tokenize_fn, batched=True)
p2_train_ds = Dataset.from_pandas(p2_train_df).map(tokenize_fn, batched=True)
p2_test_ds  = Dataset.from_pandas(p2_test_df).map(tokenize_fn, batched=True)

nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open(DEV_PATH, "r", encoding="utf-8"))
dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in spacy_doc.sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"id": doc["id"], "raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

# =========================================================================================
# 3. EVALUATION ENGINE (For Phase 1)
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text: str): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _span_token_set(token_positions, start, end): return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)
def _enrich(entities, token_positions): return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _exact_span(a, b): return a.type == b.type and a.start == b.start and a.end == b.end

def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    unmatched_b = [items_b[j] for j in range(len(items_b)) if j not in used]
    return matched, unmatched_a, unmatched_b

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        matched, unmatched_gold, unmatched_pred = _greedy_match(gold_comps, pred_comps, _exact_span)
        for g, _ in matched: tp_s[g.type] += 1
        for r in unmatched_pred: fp_s[r.type] += 1
        for r in unmatched_gold: fn_s[r.type] += 1
    f1s = []
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) else 0.0)
    return {"Macro_F1": sum(f1s) / len(f1s) if f1s else 0.0}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3
    text_to_pred = {text: pred for text, pred in zip(p1_dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities, e_id = [], 1
        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
                e_id += 1
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    return {"strict_macro_f1": evaluate_strict_f1(cases)["Macro_F1"]}

# Phase 1 Weight Calculator
counts_p1 = np.bincount(p1_train_df["label"].values, minlength=4).astype(float)
weights_p1 = ((counts_p1.sum() / (len(counts_p1) * counts_p1)) / (counts_p1.sum() / (len(counts_p1) * counts_p1)).mean()).tolist()
weights_p1[3] = 10.0

class WeightedCETrainer_P1(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p1, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# 🐛 BUG FIX: The "hasattr" guard prevents AttributeError if Trainer clones the callback!
class BestEpochTracker(TrainerCallback):
    def __init__(self):
        self.best_f1 = 0.0
        self.best_epoch = 1

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if not hasattr(self, 'best_f1'):
            self.best_f1 = 0.0
            self.best_epoch = 1
            
        current_f1 = metrics.get("eval_strict_macro_f1", 0.0)
        if current_f1 > self.best_f1:
            self.best_f1 = current_f1
            self.best_epoch = round(state.epoch)

# =========================================================================================
# 🚀 PHASE 1: SEARCH FOR BEST EPOCH
# =========================================================================================
print("\n" + "="*80)
print(f"🔍 PHASE 1: SEARCHING FOR OPTIMAL EPOCH (SEED {SEED})")
print("="*80)

model_p1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" in n]
base_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" not in n]
optim_p1 = torch.optim.AdamW([{"params": base_params_p1, "lr": 1e-5}, {"params": head_params_p1, "lr": 1e-4}], weight_decay=0.05)

args_p1 = TrainingArguments(
    output_dir="./p1_search", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=15, fp16=True,
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=SEED, data_seed=SEED
)

tracker = BestEpochTracker()
trainer_p1 = WeightedCETrainer_P1(
    model=model_p1, args=args_p1, train_dataset=p1_train_ds, eval_dataset=p1_dev_ds,
    compute_metrics=compute_metrics, callbacks=[tracker, EarlyStoppingCallback(early_stopping_patience=4)], optimizers=(optim_p1, None)
)

trainer_p1.train()

# Safe retrieval of the tracked attributes
FOUND_BEST_EPOCH = getattr(tracker, 'best_epoch', 1)
BEST_F1_SCORE = getattr(tracker, 'best_f1', 0.0)
TARGET_P2_EPOCHS = FOUND_BEST_EPOCH + EXTRA_EPOCHS_FOR_FULL_FIT

print(f"\n🎯 PHASE 1 COMPLETE! Best Epoch Found: {FOUND_BEST_EPOCH} (Score: {BEST_F1_SCORE:.4f})")
print(f"➕ Adding {EXTRA_EPOCHS_FOR_FULL_FIT} for Full-Fit. Phase 2 Target: {TARGET_P2_EPOCHS} Epochs.")

# --- MEMORY WIPE ---
del model_p1, trainer_p1, optim_p1
gc.collect()
torch.cuda.empty_cache()

# =========================================================================================
# 🚀 PHASE 2: PRODUCTION FULL-FIT & BLIND LOGIT EXTRACTION
# =========================================================================================
print("\n" + "="*80)
print(f"🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED {SEED} | {TARGET_P2_EPOCHS} EPOCHS)")
print("="*80)

# Phase 2 Weights
counts_p2 = np.bincount(p2_train_df["label"].values, minlength=4).astype(float)
weights_p2 = ((counts_p2.sum() / (len(counts_p2) * counts_p2)) / (counts_p2.sum() / (len(counts_p2) * counts_p2)).mean()).tolist()
weights_p2[3] = 10.0

class WeightedCETrainer_P2(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p2, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# Phase 2 Blind Logit Saver Callback
class BlindEpochLogitSaverCallback(TrainerCallback):
    def __init__(self, test_dataset, seed, file_prefix):
        self.test_dataset = test_dataset
        self.seed = seed
        self.file_prefix = file_prefix
        self.trainer_ref = None

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        if self.trainer_ref is not None:
            print(f"\n🔮 Extracting Blind Predictions for Phase 2, Epoch {epoch}...")
            raw_preds = self.trainer_ref.predict(self.test_dataset)
            blind_logits = raw_preds.predictions
            output_name = f"{self.file_prefix}_seed{self.seed}_epoch{epoch}_blind_logits.npy"
            np.save(output_name, blind_logits)
            print(f"💾 Saved: {output_name}")

# Init Fresh Model for Full-Fit
model_p2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" in n]
base_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" not in n]
optim_p2 = torch.optim.AdamW([{"params": base_params_p2, "lr": 1e-5}, {"params": head_params_p2, "lr": 1e-4}], weight_decay=0.05)

args_p2 = TrainingArguments(
    output_dir="./p2_production", eval_strategy="no", save_strategy="no",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16, 
    per_device_eval_batch_size=32, num_train_epochs=TARGET_P2_EPOCHS, fp16=True, 
    report_to="none", label_smoothing_factor=0.15, seed=SEED, data_seed=SEED
)

blind_saver = BlindEpochLogitSaverCallback(test_dataset=p2_test_ds, seed=SEED, file_prefix=FILE_PREFIX)

trainer_p2 = WeightedCETrainer_P2(
    model=model_p2, args=args_p2, train_dataset=p2_train_ds, 
    callbacks=[blind_saver], optimizers=(optim_p2, None)
)
blind_saver.trainer_ref = trainer_p2

trainer_p2.train()

print(f"\n🎉 TWO-STAGE RUN COMPLETE FOR SEED {SEED}! Check your output folder for the blind test .npy files.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 78.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/37.0M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

...


🎯 PHASE 1 COMPLETE! Best Epoch Found: 4 (Score: 0.7289)
➕ Adding 1 for Full-Fit. Phase 2 Target: 5 Epochs.

🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED 42 | 5 EPOCHS)


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: BSC-LT/MrBERT-biomed
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.850899
1000,0.459610
1500,0.310208



🔮 Extracting Blind Predictions for Phase 2, Epoch 1...
💾 Saved: mrbert_biomed_seed42_epoch1_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 2...
💾 Saved: mrbert_biomed_seed42_epoch2_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 3...
💾 Saved: mrbert_biomed_seed42_epoch3_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 4...
💾 Saved: mrbert_biomed_seed42_epoch4_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 5...
💾 Saved: mrbert_biomed_seed42_epoch5_blind_logits.npy

🎉 TWO-STAGE RUN COMPLETE FOR SEED 42! Check your output folder for the blind test .npy files.


In [1]:
# =========================================================================================
# 🏆 THE TWO-STAGE AUTO-RUNNER (Self-Tuning & Blind Logit Extraction)
# Model: BSC-Bio-EHR (Pure Clinical Expert)
# Strategy: Bi-Directional Context [SEP] Window
# Phase 1: Finds best epoch on Dev. 
# Phase 2: Trains on Train+Dev for (Best_Epoch + 1) and saves blind logits per epoch.
# =========================================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
import gc
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import Dataset

# --- CONFIGURATION HUB ---
SEED = 100
MODEL_NAME = "PlanTL-GOB-ES/bsc-bio-ehr-es" 
FILE_PREFIX = "bsc_bio"    
EXTRA_EPOCHS_FOR_FULL_FIT = 1       

TRAIN_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json"
DEV_PATH   = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
BLIND_TEST_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/GRACE_2026_blind_test/track_1_blind_test.json"
# -------------------------

# =========================================================================================
# 0. SEED LOCK
# =========================================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# =========================================================================================
# 1. PARSERS (Bi-Directional Context Edition)
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": sent.start_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            context_string = f"{prev_text} [SEP] {next_text}".strip()
            records.append({"text": sr["text"], "context": context_string, "label": sr["label"]})
    return pd.DataFrame(records)

def parse_blind_test(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        spacy_doc = nlp(raw_text)
        sent_records = []
        for sent in spacy_doc.sents:
            if len(sent.text.strip()) > 10:
                sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            next_text = sent_records[i + 1]["text"] if i < len(sent_records) - 1 else ""
            context_string = f"{prev_text} [SEP] {next_text}".strip()
            records.append({"text": sr["text"], "context": context_string, "label": 0})
    return pd.DataFrame(records)

# =========================================================================================
# 2. DATA PREPARATION FOR BOTH PHASES (Map before Merge)
# =========================================================================================
print("Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...")
p1_train_df = parse_grace_json(TRAIN_PATH)
p1_dev_df   = parse_grace_json(DEV_PATH)
p2_test_df  = parse_blind_test(BLIND_TEST_PATH)

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

# Ensure mapping is done BEFORE merge!
p1_train_df["label"] = p1_train_df["label"].map(label_map)
p1_dev_df["label"]   = p1_dev_df["label"].map(label_map)
p2_train_df = pd.concat([p1_train_df, p1_dev_df], ignore_index=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_fn(examples):
    # Notice we use "context" here for Bi-Directional!
    return tokenizer(examples["context"], examples["text"], padding="max_length", truncation=True, max_length=256, return_token_type_ids=False)

p1_train_ds = Dataset.from_pandas(p1_train_df).map(tokenize_fn, batched=True)
p1_dev_ds   = Dataset.from_pandas(p1_dev_df).map(tokenize_fn, batched=True)
p2_train_ds = Dataset.from_pandas(p2_train_df).map(tokenize_fn, batched=True)
p2_test_ds  = Dataset.from_pandas(p2_test_df).map(tokenize_fn, batched=True)

nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open(DEV_PATH, "r", encoding="utf-8"))
dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in spacy_doc.sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"id": doc["id"], "raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

# =========================================================================================
# 3. EVALUATION ENGINE (For Phase 1)
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text: str): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _span_token_set(token_positions, start, end): return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)
def _enrich(entities, token_positions): return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _exact_span(a, b): return a.type == b.type and a.start == b.start and a.end == b.end

def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    unmatched_b = [items_b[j] for j in range(len(items_b)) if j not in used]
    return matched, unmatched_a, unmatched_b

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        matched, unmatched_gold, unmatched_pred = _greedy_match(gold_comps, pred_comps, _exact_span)
        for g, _ in matched: tp_s[g.type] += 1
        for r in unmatched_pred: fp_s[r.type] += 1
        for r in unmatched_gold: fn_s[r.type] += 1
    f1s = []
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) else 0.0)
    return {"Macro_F1": sum(f1s) / len(f1s) if f1s else 0.0}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3
    text_to_pred = {text: pred for text, pred in zip(p1_dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities, e_id = [], 1
        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
                e_id += 1
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    return {"strict_macro_f1": evaluate_strict_f1(cases)["Macro_F1"]}

# Phase 1 Weight Calculator
counts_p1 = np.bincount(p1_train_df["label"].values, minlength=4).astype(float)
weights_p1 = ((counts_p1.sum() / (len(counts_p1) * counts_p1)) / (counts_p1.sum() / (len(counts_p1) * counts_p1)).mean()).tolist()
weights_p1[3] = 10.0

class WeightedCETrainer_P1(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p1, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# The "hasattr" guard prevents AttributeError if Trainer clones the callback!
class BestEpochTracker(TrainerCallback):
    def __init__(self):
        self.best_f1 = 0.0
        self.best_epoch = 1

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if not hasattr(self, 'best_f1'):
            self.best_f1 = 0.0
            self.best_epoch = 1
            
        current_f1 = metrics.get("eval_strict_macro_f1", 0.0)
        if current_f1 > self.best_f1:
            self.best_f1 = current_f1
            self.best_epoch = round(state.epoch)

# =========================================================================================
# 🚀 PHASE 1: SEARCH FOR BEST EPOCH
# =========================================================================================
print("\n" + "="*80)
print(f"🔍 PHASE 1: SEARCHING FOR OPTIMAL EPOCH (SEED {SEED})")
print("="*80)

model_p1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" in n]
base_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" not in n]
optim_p1 = torch.optim.AdamW([{"params": base_params_p1, "lr": 1e-5}, {"params": head_params_p1, "lr": 1e-4}], weight_decay=0.05)

args_p1 = TrainingArguments(
    output_dir="./p1_search", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", 
    per_device_train_batch_size=16,          # Fast FP16 batching!
    per_device_eval_batch_size=32, num_train_epochs=15, 
    fp16=True,                               # BSC natively supports FP16
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=SEED, data_seed=SEED
)

tracker = BestEpochTracker()
trainer_p1 = WeightedCETrainer_P1(
    model=model_p1, args=args_p1, train_dataset=p1_train_ds, eval_dataset=p1_dev_ds,
    compute_metrics=compute_metrics, callbacks=[tracker, EarlyStoppingCallback(early_stopping_patience=4)], optimizers=(optim_p1, None)
)

trainer_p1.train()

# Safe retrieval of the tracked attributes
FOUND_BEST_EPOCH = getattr(tracker, 'best_epoch', 1)
BEST_F1_SCORE = getattr(tracker, 'best_f1', 0.0)
TARGET_P2_EPOCHS = FOUND_BEST_EPOCH + EXTRA_EPOCHS_FOR_FULL_FIT

print(f"\n🎯 PHASE 1 COMPLETE! Best Epoch Found: {FOUND_BEST_EPOCH} (Score: {BEST_F1_SCORE:.4f})")
print(f"➕ Adding {EXTRA_EPOCHS_FOR_FULL_FIT} for Full-Fit. Phase 2 Target: {TARGET_P2_EPOCHS} Epochs.")

# --- MEMORY WIPE ---
del model_p1, trainer_p1, optim_p1
gc.collect()
torch.cuda.empty_cache()

# =========================================================================================
# 🚀 PHASE 2: PRODUCTION FULL-FIT & BLIND LOGIT EXTRACTION
# =========================================================================================
print("\n" + "="*80)
print(f"🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED {SEED} | {TARGET_P2_EPOCHS} EPOCHS)")
print("="*80)

# Phase 2 Weights
counts_p2 = np.bincount(p2_train_df["label"].values, minlength=4).astype(float)
weights_p2 = ((counts_p2.sum() / (len(counts_p2) * counts_p2)) / (counts_p2.sum() / (len(counts_p2) * counts_p2)).mean()).tolist()
weights_p2[3] = 10.0

class WeightedCETrainer_P2(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p2, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# Phase 2 Blind Logit Saver Callback
class BlindEpochLogitSaverCallback(TrainerCallback):
    def __init__(self, test_dataset, seed, file_prefix):
        self.test_dataset = test_dataset
        self.seed = seed
        self.file_prefix = file_prefix
        self.trainer_ref = None

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        if self.trainer_ref is not None:
            print(f"\n🔮 Extracting Blind Predictions for Phase 2, Epoch {epoch}...")
            raw_preds = self.trainer_ref.predict(self.test_dataset)
            blind_logits = raw_preds.predictions
            output_name = f"{self.file_prefix}_seed{self.seed}_epoch{epoch}_blind_logits.npy"
            np.save(output_name, blind_logits)
            print(f"💾 Saved: {output_name}")

# Init Fresh Model for Full-Fit
model_p2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" in n]
base_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" not in n]
optim_p2 = torch.optim.AdamW([{"params": base_params_p2, "lr": 1e-5}, {"params": head_params_p2, "lr": 1e-4}], weight_decay=0.05)

args_p2 = TrainingArguments(
    output_dir="./p2_production", eval_strategy="no", save_strategy="no",
    warmup_steps=100, lr_scheduler_type="cosine", 
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=32, num_train_epochs=TARGET_P2_EPOCHS, 
    fp16=True, 
    report_to="none", label_smoothing_factor=0.15, seed=SEED, data_seed=SEED
)

blind_saver = BlindEpochLogitSaverCallback(test_dataset=p2_test_ds, seed=SEED, file_prefix=FILE_PREFIX)

trainer_p2 = WeightedCETrainer_P2(
    model=model_p2, args=args_p2, train_dataset=p2_train_ds, 
    callbacks=[blind_saver], optimizers=(optim_p2, None)
)
blind_saver.trainer_ref = trainer_p2

trainer_p2.train()

print(f"\n🎉 TWO-STAGE RUN COMPLETE FOR SEED {SEED}! Check your output folder for the blind test .npy files.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 78.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


🎯 PHASE 1 COMPLETE! Best Epoch Found: 4 (Score: 0.6664)
➕ Adding 1 for Full-Fit. Phase 2 Target: 5 Epochs.

🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED 42 | 5 EPOCHS)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: PlanTL-GOB-ES/bsc-bio-ehr-es
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.716889
1000,0.457648
1500,0.303010



🔮 Extracting Blind Predictions for Phase 2, Epoch 1...
💾 Saved: bsc_bio_seed42_epoch1_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 2...
💾 Saved: bsc_bio_seed42_epoch2_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 3...
💾 Saved: bsc_bio_seed42_epoch3_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 4...
💾 Saved: bsc_bio_seed42_epoch4_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 5...
💾 Saved: bsc_bio_seed42_epoch5_blind_logits.npy

🎉 TWO-STAGE RUN COMPLETE FOR SEED 42! Check your output folder for the blind test .npy files.


In [6]:
import numpy as np
import json
import torch
import torch.nn.functional as F
import spacy

print("Initiating Final Submission Generator for MrBERT...")

# 1. LOAD THE MATHEMATICALLY OPTIMAL EPOCH
# (Phase 1 peaked at 4, so on Train+Dev, Epoch 5 is our target)
LOGITS_FILE = "/kaggle/input/datasets/alexcristea72/logits/mrbert_biomed_seed42_epoch4_blind_logits.npy"
logits = np.load(LOGITS_FILE)

# Convert raw logits to clean probabilities (0.0 to 1.0)
probs = F.softmax(torch.tensor(logits), dim=-1).numpy()

# Get the highest probability class
final_preds = np.argmax(probs, axis=-1)

# Apply the Grandmaster MajorClaim rule (if >50% sure it's a MajorClaim, force it!)
final_preds[probs[:, 3] > 0.50] = 3 

# 2. BUILD THE OFFICIAL SUBMISSION JSON
BLIND_JSON_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/GRACE_2026_blind_test/track_1_blind_test.json"
with open(BLIND_JSON_PATH, "r", encoding="utf-8") as f: 
    blind_data = json.load(f)

# 🐛 FIXED: Load standard spaCy to guarantee exact sentence alignment with training!
nlp = spacy.load("es_core_news_sm")

inv_label_map = {0: "None", 1: "Premise", 2: "Claim", 3: "MajorClaim"}
submission = []
flat_idx = 0 

print("Mapping predictions back to Blind Test sentences...")
for doc in blind_data:
    raw_text = doc["raw_text"]
    ents = []
    e_id = 1
    
    # We iterate sentences exactly the same way we did during Data Prep
    for sent in nlp(raw_text).sents:
        if len(sent.text.strip()) > 10:
            lbl = inv_label_map[final_preds[flat_idx]]
            if lbl != "None":
                ents.append({
                    "id": f"T{e_id}", 
                    "text": sent.text.strip(), 
                    "start": sent.start_char, 
                    "end": sent.end_char, 
                    "type": lbl
                })
                e_id += 1
            flat_idx += 1
            
    submission.append({
        "id": doc["id"], 
        "raw_text": raw_text, 
        "metadata": doc.get("metadata", {}),
        "annotations": {"entities": ents, "relations": []}, # Official format expects both
        "predictions": {"entities": ents, "relations": []}
    })

# 3. SAVE AND EXPORT
output_file = "MRBERT_SOLO_SUBMISSION_eP_4.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=4)

print(f"🎉 BOOM! File ready: {output_file}.")
print("You can now download this and submit it directly to CodaBench!")

Initiating Final Submission Generator for MrBERT...
Mapping predictions back to Blind Test sentences...
🎉 BOOM! File ready: MRBERT_SOLO_SUBMISSION_eP_4.json.
You can now download this and submit it directly to CodaBench!


In [ ]:
# =========================================================================================
# 🏆 THE TWO-STAGE AUTO-RUNNER (Self-Tuning & Blind Logit Extraction)
# Phase 1: Finds best epoch on Dev. 
# Phase 2: Trains on Train+Dev for (Best_Epoch + 1) and saves blind logits per epoch.
# =========================================================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess, sys
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"], check=False)

import random
import json
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import spacy
import gc
from collections import defaultdict
from typing import NamedTuple

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import Dataset

# --- CONFIGURATION HUB ---
SEED = 24
MODEL_NAME = "BSC-LT/MrBERT-biomed" # Change to assigned model
FILE_PREFIX = "mrbert_biomed"       # E.g., mrbert_biomed, deberta, bsc_bio
EXTRA_EPOCHS_FOR_FULL_FIT = 1       # Adds +1 to the best found epoch

TRAIN_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_train.json"
DEV_PATH   = "/kaggle/input/datasets/alexcristea72/iberlef-grace/IberLEF_GRACE/public_data/track_1_dev.json"
TRAIN_DEV_PATH = "merged_train_dev_full_fit.json"
BLIND_TEST_PATH = "/kaggle/input/datasets/alexcristea72/iberlef-grace/GRACE_2026_blind_test/track_1_blind_test.json"
# -------------------------

# =========================================================================================
# 0. SEED LOCK
# =========================================================================================
def seed_everything(seed=24):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# =========================================================================================
# 1. PARSERS
# =========================================================================================
def parse_grace_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        annotations = doc.get("annotations", {}).get("entities", [])
        annotated_spans = [(ann["start"], ann["end"]) for ann in annotations]
        spacy_doc = nlp(raw_text)
        sent_records = [{"text": ann["text"].strip(), "label": ann["type"], "start": ann["start"]} for ann in annotations]
        for sent in spacy_doc.sents:
            if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans):
                if len(sent.text.strip()) > 10:
                    sent_records.append({"text": sent.text.strip(), "label": "None", "start": sent.start_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": sr["label"]})
    return pd.DataFrame(records)

def parse_blind_test(file_path):
    with open(file_path, "r", encoding="utf-8") as f: data = json.load(f)
    nlp = spacy.load("es_core_news_sm")
    records = []
    for doc in data:
        raw_text = doc["raw_text"]
        spacy_doc = nlp(raw_text)
        sent_records = []
        for sent in spacy_doc.sents:
            if len(sent.text.strip()) > 10:
                sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
        sent_records.sort(key=lambda x: x["start"])
        for i, sr in enumerate(sent_records):
            prev_text = sent_records[i - 1]["text"] if i > 0 else ""
            records.append({"text": sr["text"], "prev_text": prev_text, "label": 0})
    return pd.DataFrame(records)

# =========================================================================================
# 2. DATA PREPARATION FOR BOTH PHASES (BUG FIXED: In-Memory Merge)
# =========================================================================================
print("Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...")
p1_train_df = parse_grace_json(TRAIN_PATH)
p1_dev_df   = parse_grace_json(DEV_PATH)
p2_test_df  = parse_blind_test(BLIND_TEST_PATH)

label_map = {"None": 0, "Premise": 1, "Claim": 2, "MajorClaim": 3}
inv_label_map = {v: k for k, v in label_map.items()}

# Ensure mapping is done BEFORE merge!
p1_train_df["label"] = p1_train_df["label"].map(label_map)
p1_dev_df["label"]   = p1_dev_df["label"].map(label_map)
p2_train_df = pd.concat([p1_train_df, p1_dev_df], ignore_index=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_fn(examples):
    return tokenizer(examples["prev_text"], examples["text"], padding="max_length", truncation=True, max_length=192, return_token_type_ids=False)

p1_train_ds = Dataset.from_pandas(p1_train_df).map(tokenize_fn, batched=True)
p1_dev_ds   = Dataset.from_pandas(p1_dev_df).map(tokenize_fn, batched=True)
p2_train_ds = Dataset.from_pandas(p2_train_df).map(tokenize_fn, batched=True)
p2_test_ds  = Dataset.from_pandas(p2_test_df).map(tokenize_fn, batched=True)

nlp_cache = spacy.load("es_core_news_sm")
dev_raw_data = json.load(open(DEV_PATH, "r", encoding="utf-8"))
dev_eval_template = []
for doc in dev_raw_data:
    raw_text, annotations = doc["raw_text"], doc.get("annotations", {}).get("entities", [])
    annotated_spans = [(a["start"], a["end"]) for a in annotations]
    spacy_doc = nlp_cache(raw_text)
    sent_records = [{"text": a["text"].strip(), "start": a["start"], "end": a["end"]} for a in annotations]
    for sent in spacy_doc.sents:
        if not any(max(sent.start_char, a) < min(sent.end_char, b) for a, b in annotated_spans) and len(sent.text.strip()) > 10:
            sent_records.append({"text": sent.text.strip(), "start": sent.start_char, "end": sent.end_char})
    sent_records.sort(key=lambda x: x["start"])
    dev_eval_template.append({"id": doc["id"], "raw_text": raw_text, "annotations": {"entities": annotations}, "sent_records": sent_records})

# =========================================================================================
# 3. EVALUATION ENGINE (For Phase 1)
# =========================================================================================
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def _tokenize(text: str): return [(m.start(), m.end()) for m in TOKEN_RE.finditer(text)]
def _span_token_set(token_positions, start, end): return frozenset(i for i, (ts, te) in enumerate(token_positions) if ts >= start and te <= end)
def _enrich(entities, token_positions): return [{**e, "tokens": _span_token_set(token_positions, e["start"], e["end"])} for e in entities]
class CompTuple(NamedTuple): start: int; end: int; type: str; tokens: frozenset
def _exact_span(a, b): return a.type == b.type and a.start == b.start and a.end == b.end

def _greedy_match(items_a, items_b, match_fn):
    used, matched, unmatched_a = set(), [], []
    for a in items_a:
        j = next((j for j, b in enumerate(items_b) if j not in used and match_fn(a, b)), None)
        if j is not None: matched.append((a, items_b[j])); used.add(j)
        else: unmatched_a.append(a)
    unmatched_b = [items_b[j] for j in range(len(items_b)) if j not in used]
    return matched, unmatched_a, unmatched_b

def evaluate_strict_f1(cases):
    tp_s, fp_s, fn_s = defaultdict(int), defaultdict(int), defaultdict(int)
    for case in cases:
        tp = _tokenize(case.get("raw_text", ""))
        gold_ents, pred_ents = _enrich(case["annotations"]["entities"], tp), _enrich(case["predictions"]["entities"], tp)
        gold_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in gold_ents]
        pred_comps = [CompTuple(e["start"], e["end"], e["type"], e["tokens"]) for e in pred_ents]
        matched, unmatched_gold, unmatched_pred = _greedy_match(gold_comps, pred_comps, _exact_span)
        for g, _ in matched: tp_s[g.type] += 1
        for r in unmatched_pred: fp_s[r.type] += 1
        for r in unmatched_gold: fn_s[r.type] += 1
    f1s = []
    for t in ["Premise", "Claim", "MajorClaim"]:
        p = tp_s[t] / (tp_s[t] + fp_s[t]) if (tp_s[t] + fp_s[t]) else 0.0
        r = tp_s[t] / (tp_s[t] + fn_s[t]) if (tp_s[t] + fn_s[t]) else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) else 0.0)
    return {"Macro_F1": sum(f1s) / len(f1s) if f1s else 0.0}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    preds[probs[:, 3] > 0.50] = 3
    text_to_pred = {text: pred for text, pred in zip(p1_dev_df["text"], preds)}
    cases = []
    for template in dev_eval_template:
        predicted_entities, e_id = [], 1
        for sr in template["sent_records"]:
            p_label = inv_label_map.get(text_to_pred.get(sr["text"], 0), "None")
            if p_label != "None":
                predicted_entities.append({"id": f"T{e_id}", "text": sr["text"], "start": sr["start"], "end": sr["end"], "type": p_label})
                e_id += 1
        cases.append({"raw_text": template["raw_text"], "annotations": template["annotations"], "predictions": {"entities": predicted_entities}})
    return {"strict_macro_f1": evaluate_strict_f1(cases)["Macro_F1"]}

# Phase 1 Weight Calculator
counts_p1 = np.bincount(p1_train_df["label"].values, minlength=4).astype(float)
weights_p1 = ((counts_p1.sum() / (len(counts_p1) * counts_p1)) / (counts_p1.sum() / (len(counts_p1) * counts_p1)).mean()).tolist()
weights_p1[3] = 10.0

class WeightedCETrainer_P1(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p1, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# 🐛 BUG FIX: The "hasattr" guard prevents AttributeError if Trainer clones the callback!
class BestEpochTracker(TrainerCallback):
    def __init__(self):
        self.best_f1 = 0.0
        self.best_epoch = 1

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if not hasattr(self, 'best_f1'):
            self.best_f1 = 0.0
            self.best_epoch = 1
            
        current_f1 = metrics.get("eval_strict_macro_f1", 0.0)
        if current_f1 > self.best_f1:
            self.best_f1 = current_f1
            self.best_epoch = round(state.epoch)

# =========================================================================================
# 🚀 PHASE 1: SEARCH FOR BEST EPOCH
# =========================================================================================
print("\n" + "="*80)
print(f"🔍 PHASE 1: SEARCHING FOR OPTIMAL EPOCH (SEED {SEED})")
print("="*80)

model_p1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" in n]
base_params_p1 = [p for n, p in model_p1.named_parameters() if "classifier" not in n]
optim_p1 = torch.optim.AdamW([{"params": base_params_p1, "lr": 1e-5}, {"params": head_params_p1, "lr": 1e-4}], weight_decay=0.05)

args_p1 = TrainingArguments(
    output_dir="./p1_search", eval_strategy="epoch", save_strategy="epoch",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=15, fp16=True,
    load_best_model_at_end=True, metric_for_best_model="strict_macro_f1", greater_is_better=True,
    save_total_limit=1, save_only_model=True, report_to="none", label_smoothing_factor=0.15,
    seed=SEED, data_seed=SEED
)

tracker = BestEpochTracker()
trainer_p1 = WeightedCETrainer_P1(
    model=model_p1, args=args_p1, train_dataset=p1_train_ds, eval_dataset=p1_dev_ds,
    compute_metrics=compute_metrics, callbacks=[tracker, EarlyStoppingCallback(early_stopping_patience=4)], optimizers=(optim_p1, None)
)

trainer_p1.train()

# Safe retrieval of the tracked attributes
FOUND_BEST_EPOCH = getattr(tracker, 'best_epoch', 1)
BEST_F1_SCORE = getattr(tracker, 'best_f1', 0.0)
TARGET_P2_EPOCHS = FOUND_BEST_EPOCH + EXTRA_EPOCHS_FOR_FULL_FIT

print(f"\n🎯 PHASE 1 COMPLETE! Best Epoch Found: {FOUND_BEST_EPOCH} (Score: {BEST_F1_SCORE:.4f})")
print(f"➕ Adding {EXTRA_EPOCHS_FOR_FULL_FIT} for Full-Fit. Phase 2 Target: {TARGET_P2_EPOCHS} Epochs.")

# --- MEMORY WIPE ---
del model_p1, trainer_p1, optim_p1
gc.collect()
torch.cuda.empty_cache()

# =========================================================================================
# 🚀 PHASE 2: PRODUCTION FULL-FIT & BLIND LOGIT EXTRACTION
# =========================================================================================
print("\n" + "="*80)
print(f"🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED {SEED} | {TARGET_P2_EPOCHS} EPOCHS)")
print("="*80)

# Phase 2 Weights
counts_p2 = np.bincount(p2_train_df["label"].values, minlength=4).astype(float)
weights_p2 = ((counts_p2.sum() / (len(counts_p2) * counts_p2)) / (counts_p2.sum() / (len(counts_p2) * counts_p2)).mean()).tolist()
weights_p2[3] = 10.0

class WeightedCETrainer_P2(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = torch.tensor(weights_p2, dtype=outputs.logits.dtype, device=outputs.logits.device)
        return (F.cross_entropy(outputs.logits, labels, weight=w), outputs) if return_outputs else F.cross_entropy(outputs.logits, labels, weight=w)

# Phase 2 Blind Logit Saver Callback
class BlindEpochLogitSaverCallback(TrainerCallback):
    def __init__(self, test_dataset, seed, file_prefix):
        self.test_dataset = test_dataset
        self.seed = seed
        self.file_prefix = file_prefix
        self.trainer_ref = None

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        if self.trainer_ref is not None:
            print(f"\n🔮 Extracting Blind Predictions for Phase 2, Epoch {epoch}...")
            raw_preds = self.trainer_ref.predict(self.test_dataset)
            blind_logits = raw_preds.predictions
            output_name = f"{self.file_prefix}_seed{self.seed}_epoch{epoch}_blind_logits.npy"
            np.save(output_name, blind_logits)
            print(f"💾 Saved: {output_name}")

# Init Fresh Model for Full-Fit
model_p2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4, ignore_mismatched_sizes=True)
head_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" in n]
base_params_p2 = [p for n, p in model_p2.named_parameters() if "classifier" not in n]
optim_p2 = torch.optim.AdamW([{"params": base_params_p2, "lr": 1e-5}, {"params": head_params_p2, "lr": 1e-4}], weight_decay=0.05)

args_p2 = TrainingArguments(
    output_dir="./p2_production", eval_strategy="no", save_strategy="no",
    warmup_steps=100, lr_scheduler_type="cosine", per_device_train_batch_size=16, 
    per_device_eval_batch_size=32, num_train_epochs=TARGET_P2_EPOCHS, fp16=True, 
    report_to="none", label_smoothing_factor=0.15, seed=SEED, data_seed=SEED
)

blind_saver = BlindEpochLogitSaverCallback(test_dataset=p2_test_ds, seed=SEED, file_prefix=FILE_PREFIX)

trainer_p2 = WeightedCETrainer_P2(
    model=model_p2, args=args_p2, train_dataset=p2_train_ds, 
    callbacks=[blind_saver], optimizers=(optim_p2, None)
)
blind_saver.trainer_ref = trainer_p2

trainer_p2.train()

print(f"\n🎉 TWO-STAGE RUN COMPLETE FOR SEED {SEED}! Check your output folder for the blind test .npy files.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 84.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading Datasets for Phase 1 (Train/Dev) and Phase 2 (Merged/Blind)...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/37.0M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

...


🎯 PHASE 1 COMPLETE! Best Epoch Found: 6 (Score: 0.7204)
➕ Adding 1 for Full-Fit. Phase 2 Target: 7 Epochs.

🏭 PHASE 2: FULL-FIT PRODUCTION RUN (SEED 24 | 7 EPOCHS)


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: BSC-LT/MrBERT-biomed
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.829459



🔮 Extracting Blind Predictions for Phase 2, Epoch 1...
💾 Saved: mrbert_biomed_seed24_epoch1_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 2...
💾 Saved: mrbert_biomed_seed24_epoch2_blind_logits.npy

🔮 Extracting Blind Predictions for Phase 2, Epoch 3...
